# **wiki2shards**

## Phase 0: Workbench (M0)

### Step 0.1: Create the skeleton and pin Python

```py
uv python pin 3.12.14
```
- The command `uv python pin 3.12.14` sets Python version 3.12.14 as the active Python version for your current working directory and project.
- Creates/Updates `.python-version`: It writes `3.12.14` to a `.python-version` file in the current directory.
- According to Charlie Marsh (creator of uv and founder of Astral), uv officially stands for "Ultraviolet".

```sh
mkdir -p src/wiki2shards/{io,acquire,extract,clean,tokenize,pack,loader,pipeline}
```
- `-p` stands for `parent` used in parent directory

```sh
touch src/wiki2shards/{io,acquire,extract,clean,tokenize,pack,loader,pipeline}/__init__.py
```

- creates a `__init__.py` inside every single directory within the curly brackets

```sh
echo "# wiki2shards" > README.md
```
- writes `# wiki2shards` into `README.md`

### Step 0.2: pyproject.toml

`pyproject.toml`
- `pyproject.toml` is the modern standard configuration file for Python projects. It centralizes project metadata, dependencies, and configuration for various development tools into a single file, replacing the older, fragmented ecosystem of setup.py, setup.cfg, and requirements.txt.

**`[project]`**
Declares the start of the core, standardized project metadata block (defined by PEP 621).

* **`name = "wiki2shards"`**
Sets the package's name. This is what you would use to pip install it.
* **`version = "0.1.0"`**
Sets the current version of the project.
* **`description = "Wikipedia dump -> checksummed token shards, plus a deterministic, resumable, mixture-aware loader"`**
Provides a short summary of the project's purpose.
* **`readme = "README.md"`**
Specifies that the long description (usually displayed on PyPI) should be pulled from the `README.md` file.
* **`requires-python = ">=3.12,<3.13"`**
Restricts this package to only run on Python 3.12.x. It will fail to install on Python 3.11 or Python 3.13.
* **`dependencies = [`**
Starts the list of core runtime dependencies required for the package to function.
* **`"numpy>=2.0",`**
Requires NumPy version 2.0 or higher.
* **`"mwparserfromhell==0.7.2",  # output bytes depend on it (wikitext goldens)`**
Pins this Wikipedia parsing library to exactly version 0.7.2. The comment explains this strict pinning ensures the processed output remains identical across builds.
* **`"tiktoken>=0.9",`**
Requires the `tiktoken` (OpenAI tokenizer) library version 0.9 or higher.
* **`"tokenizers==0.23.3",  # output bytes depend on it (fixture tokenizer)`**
Pins Hugging Face `tokenizers` to exactly 0.23.3 to guarantee deterministic tokenization.
* **`"pydantic>=2.7",`**
Requires Pydantic version 2.7 or higher for data validation.
* **`"typer>=0.12",`**
Requires Typer version 0.12 or higher to build the command-line interface.
* **`"pyyaml>=6.0",`**
Requires PyYAML version 6.0 or higher to parse YAML files.
* **`]`**
Closes the core dependencies list.

**`[project.scripts]`**
Defines console commands (CLI entry points) that get automatically installed with the package.

* **`wiki2shards = "wiki2shards.cli:main"`**
Creates a terminal command named `wiki2shards` that, when run, executes the `main` function inside the `wiki2shards.cli` Python module.

**`[dependency-groups]`**
Defines standard dependency groups (PEP 735) for different environments, natively supported by tools like `uv`.

* **`dev = [`**
Starts a group named `dev` containing dependencies only needed for developing the package, not running it.
* **`"pytest>=8",`**
Requires the Pytest testing framework version 8 or higher.
* **`"hypothesis>=6.100",`**
Requires Hypothesis v6.100+ for property-based testing.
* **`"pytest-timeout>=2.3",`**
Requires the pytest-timeout plugin v2.3+ to forcibly stop tests that hang.
* **`"ruff>=0.6",`**
Requires the Ruff linter/formatter version 0.6 or higher.
* **`"mypy>=1.11",`**
Requires the Mypy static type checker version 1.11 or higher.
* **`"types-PyYAML>=6.0",`**
Installs type hints for PyYAML so Mypy can properly type-check YAML-related code.
* **`]`**
Closes the dev dependencies list.

**`[build-system]`**
Defines the tools pip needs to build the package from source into a wheel (PEP 518).

* **`requires = ["hatchling"]`**
Specifies that the `hatchling` package must be downloaded before the build can start.
* **`build-backend = "hatchling.build"`**
Tells the build frontend (like pip) to use Hatchling's build system to package the code.

**`[tool.hatch.build.targets.wheel]`**
Configuration specific to the Hatchling build backend regarding how to build the `.whl` (wheel) file.

* **`packages = ["src/wiki2shards"]`**
Tells Hatchling that the actual Python code to be packaged lives in the `src/wiki2shards` directory (a standard `src` layout).

**`[tool.pytest.ini_options]`**
Configuration block for the Pytest testing framework.

* **`testpaths = ["tests"]`**
Tells Pytest to only look for tests inside the `tests` directory.
* **`pythonpath = ["tests"]`**
Adds the `tests` directory to the Python path during a test run, allowing test files to import helper functions from other test files.
* **`addopts = "-p no:cacheprovider --strict-markers"`**
Default command-line flags applied to every pytest run: disables test result caching (`-p no:cacheprovider`) and throws an error if an unregistered test marker is used (`--strict-markers`).
* **`timeout = 120`**
Instructs `pytest-timeout` to automatically fail any single test that takes longer than 120 seconds.
* **`markers = [`**
Starts a list to register custom test markers (decorators like `@pytest.mark.slow`).
* **`"slow: subprocess builds, crash matrix, SIGKILL resume (excluded from the fast suite)",`**
Registers the `slow` marker and provides a description of what it's used for.
* **`"bench: benchmarks, never gating",`**
Registers the `bench` marker used for benchmarking tests.
* **`]`**
Closes the markers list.

**`[tool.ruff]`**
Configuration block for the Ruff linter and formatter.

* **`line-length = 100`**
Sets the maximum allowed characters per line to 100 (overriding the default 88).
* **`target-version = "py312"`**
Tells Ruff to assume the code is running on Python 3.12, allowing it to apply modern formatting and linting rules specific to that version.

**`[tool.ruff.lint]`**
Specific configuration for Ruff's linting features.

* **`select = ["E", "F", "W", "I", "B", "UP", "SIM", "RUF"]`**
Enables specific families of linting rules: `E` and `W` (pycodestyle errors/warnings), `F` (Pyflakes), `I` (isort for imports), `B` (flake8-bugbear for common bugs), `UP` (pyupgrade for modern syntax), `SIM` (flake8-simplify), and `RUF` (Ruff's own rules).

**`[tool.mypy]`**
Configuration block for the Mypy static type checker.

* **`strict = true`**
Turns on Mypy's most restrictive mode, requiring type hints for everything and prohibiting dynamic typing loopholes.
* **`python_version = "3.12"`**
Tells Mypy to type-check against Python 3.12 rules.
* **`files = ["src"]`**
Instructs Mypy to only check the code inside the `src` directory (ignoring tests or scripts).

**`[[tool.mypy.overrides]]`**
Starts an override block (denoted by double brackets `[[ ]]` representing an array of tables in TOML) to apply specific Mypy rules to specific libraries.

* **`module = ["mwparserfromhell", "mwparserfromhell.*"]`**
Targets the `mwparserfromhell` library and all of its submodules for this override.
* **`ignore_missing_imports = true`**
Tells Mypy to suppress errors about this library missing type hints (since it does not provide them natively and no stub package exists).

### Step 0.3: Package marker, then install

`src/wiki2shards/__init__.py`

### Step 0.4: Crash-safe writes

`src/wiki2shards/io/atomic.py`

```python
"""Crash-safe writes: tmp -> fsync -> os.replace -> fsync(dir).

A reader can never observe a partially written file under its final name: the bytes land in a
hidden tmp file next to it, and only a complete, fsynced file is renamed into place. os.replace
is atomic on POSIX within one filesystem, which is why the tmp file lives in the same directory.
"""

```

* **Lines 1-6:** A module-level docstring summarizing the entire script's purpose. It explains the core mechanism of atomic file writes: data is written to a temporary file, synced to disk, and then atomically renamed over the original file so no other program ever sees a partially written state.

```python
from __future__ import annotations

```

* **Line 8:** Instructs Python to delay evaluating type hints. This allows you to use modern type hint syntax (like `Path | str`) even in older Python versions, and prevents circular import issues with typing.

```python
import os
from collections.abc import Iterator
from contextlib import contextmanager
from pathlib import Path
from typing import BinaryIO

```

* **Line 10:** Imports the standard `os` module, providing low-level operating system interfaces (like `fsync`, `replace`, `open`, `close`).
* **Line 11:** Imports `Iterator` for type hinting the return type of the context manager.
* **Line 12:** Imports the `contextmanager` decorator, which allows writing a function that behaves like a class with `__enter__` and `__exit__` methods for use in a `with` statement.
* **Line 13:** Imports `Path` from `pathlib` for modern, object-oriented filesystem path manipulation.
* **Line 14:** Imports `BinaryIO` for type hinting a file object opened in binary mode.

```python
TMP_SUFFIX = ".tmp"

```

* **Line 16:** Defines a constant string `.tmp` that will be appended to the end of temporary files.

```python
def tmp_name(final: Path) -> Path:
    """Hidden sibling used while `final` is being written."""
    return final.with_name(f".{final.name}{TMP_SUFFIX}")

```

* **Line 19:** Defines a function `tmp_name` that takes a `Path` object (the target file) and returns a new `Path` object.
* **Line 20:** A docstring explaining the function's purpose.
* **Line 21:** Generates the temporary file path. It takes the original filename (`final.name`), prepends a dot `.` to make it a hidden file on UNIX/Linux systems, and appends the `.tmp` suffix. It replaces the original filename in the original directory (`final.with_name(...)`).

```python
def fsync_dir(path: Path) -> None:
    """Persist a directory entry (the rename) itself."""
    fd = os.open(path, os.O_RDONLY)
    try:
        os.fsync(fd)
    finally:
        os.close(fd)

```

* **Line 24:** Defines a function `fsync_dir` that takes a directory path and returns nothing (`None`).
* **Line 25:** A docstring explaining that this persists the directory metadata (necessary to guarantee a file rename survives a power loss).
* **Line 26:** Uses the low-level `os.open` to open the directory itself in read-only mode (`os.O_RDONLY`), returning an integer file descriptor (`fd`).
* **Line 27:** Starts a `try` block to ensure cleanup happens even if an error occurs.
* **Line 28:** Calls `os.fsync()` on the directory's file descriptor, forcing the operating system to flush the directory's metadata changes to physical storage.
* **Line 29:** Starts the `finally` block, which executes regardless of whether the `try` block succeeded or threw an exception.
* **Line 30:** Closes the low-level file descriptor to prevent resource leaks.

```python
@contextmanager
def atomic_open(final: Path | str) -> Iterator[BinaryIO]:
    """Open a binary writer whose content appears at `final` only if the block succeeds."""

```

* **Line 33:** Applies the `@contextmanager` decorator, converting the generator function below into a context manager that can be used with a `with` statement.
* **Line 34:** Defines the `atomic_open` function, which accepts either a `Path` or a string, and yields a binary file object (`BinaryIO`).
* **Line 35:** A docstring explaining that the file will only appear at its final destination if the code block inside the `with` statement completes without errors.

```python
    final = Path(final)
    tmp = tmp_name(final)
    f = open(tmp, "wb")  # noqa: SIM115 - closed explicitly on both paths below

```

* **Line 36:** Normalizes the input `final` by ensuring it is a `Path` object (in case a string was passed).
* **Line 37:** Calls `tmp_name()` to determine the hidden temporary file path.
* **Line 38:** Opens the temporary file for writing in binary mode (`"wb"`). The `# noqa: SIM115` comment tells a specific linter (flake8-simplify) to ignore the warning about opening a file without using a `with` statement, since this function manually manages the closing.

```python
    try:
        yield f
        f.flush()
        os.fsync(f.fileno())
        f.close()
        os.replace(tmp, final)
        fsync_dir(final.parent)

```

* **Line 39:** Starts the `try` block for the "success" path.
* **Line 40:** `yield f` hands the open temporary file object back to the caller's `with` block. Execution pauses here until the user's `with` block finishes.
* **Line 41:** Once the user's block finishes successfully, execution resumes. `f.flush()` forces Python's internal memory buffers to write to the OS.
* **Line 42:** `os.fsync(f.fileno())` tells the OS to physically write its own disk buffers to the hardware storage, guaranteeing the data is saved.
* **Line 43:** Closes the temporary file. It is now safely written to disk, but still has the `.tmp` name.
* **Line 44:** `os.replace` atomically renames the temporary file to the final filename. If a file already exists at `final`, it is silently and atomically overwritten.
* **Line 45:** Calls the previously defined `fsync_dir` on the parent directory (`final.parent`) to guarantee that the OS persists the file rename to disk.

```python
    except BaseException:
        f.close()
        tmp.unlink(missing_ok=True)
        raise

```

* **Line 46:** The `except BaseException` block catches *any* exception that occurred while the user was interacting with the yielded file in the `with` block, including system-exiting exceptions like `KeyboardInterrupt`.
* **Line 47:** Closes the file object to release the resource.
* **Line 48:** Deletes the temporary file (`unlink`). `missing_ok=True` prevents a `FileNotFoundError` if the file somehow doesn't exist.
* **Line 49:** `raise` re-throws the original exception so the calling program knows an error occurred and can handle it or crash normally.

```python
def atomic_write_bytes(final: Path | str, data: bytes) -> None:
    with atomic_open(final) as f:
        f.write(data)

```

* **Line 52:** Defines a helper function that takes a target path and a payload of raw bytes, returning nothing.
* **Line 53:** Uses the `atomic_open` context manager defined above to safely open the destination file.
* **Line 54:** Writes the entire `data` byte payload into the yielded file object. Once this block ends, `atomic_open` handles flushing, syncing, and renaming.

```python
def commit(src: Path, final: Path) -> None:
    """Move an already-complete file (written elsewhere in the same filesystem) into place."""
    with open(src, "rb") as f:
        os.fsync(f.fileno())
    os.replace(src, final)
    fsync_dir(final.parent)

```

* **Line 57:** Defines a `commit` function intended to take an existing file (`src`) and safely finalize it as `final`.
* **Line 58:** A docstring explaining its use case.
* **Line 59:** Opens the source file in read-binary mode using a standard `with` context manager.
* **Line 60:** Calls `os.fsync()` on the source file to ensure any pending writes made by whoever created the file are physically on disk before moving it.
* **Line 61:** The `with` block ends, automatically closing the source file.
* **Line 62:** Atomically renames the source file to its final destination.
* **Line 63:** Syncs the destination's parent directory to ensure the rename operation is permanently recorded on disk.

### Step 0.5: Hashing, canonical JSON and document identity

`src/wiki2shards/io/hashing.py`

> File Header & Setup

* `"""Stable hashing, canonical JSON and document identity.` — The start of a multi-line comment describing what this file does.
* `Nothing here uses the built-in hash(): str hashing is salted per process (PYTHONHASHSEED), so any` — Explains that Python's default tool for mixing data (`hash()`) changes every time you restart the program.
* `value derived from it would differ between runs. Everything persisted goes through sha256.` — Because default hashes change, this code uses `sha256` (a standard mathematical formula) to ensure digital fingerprints stay exactly the same forever.
* `"""` — The end of the multi-line comment.
* `from __future__ import annotations` — Allows the code to use modern Python type hints (labels telling you what type of data a variable is) even on older Python versions.
* `import hashlib` — Brings in Python's built-in tool for creating secure digital fingerprints (hashes).
* `import json` — Brings in the tool to read and write JSON, a very common text format used to store and send data.
* `from pathlib import Path` — Brings in a tool that makes it easy to work with computer file and folder paths.
* `from typing import Any` — Brings in a special label (`Any`) used to tell the code "this variable can hold absolutely any type of data."
* `_CHUNK = 1 << 20` — Creates a constant variable set to 1,048,576 (which is exactly 1 Megabyte). This is used to read large files in small pieces so they don't crash the computer's memory.

> Hashing Functions (Making Digital Fingerprints)

* `def sha256_hex(data: bytes) -> str:` — Defines a function named `sha256_hex` that takes raw computer bytes and returns a standard text string.
* `return hashlib.sha256(data).hexdigest()` — Creates a SHA-256 fingerprint of the bytes and hands it back as readable text (hexadecimal).
* `def sha256_file(path: Path | str) -> str:` — Defines a function that takes a file path (as a Path object or text string) and returns a text string.
* `h = hashlib.sha256()` — Starts an empty SHA-256 fingerprint calculator.
* `with open(path, "rb") as f:` — Safely opens the file in "read binary" (`rb`) mode so it can read the raw 1s and 0s.
* `while chunk := f.read(_CHUNK):` — A loop that reads the file one 1-Megabyte chunk at a time until the file is empty.
* `h.update(chunk)` — Adds that 1-Megabyte chunk into the fingerprint calculator.
* `return h.hexdigest()` — Once the whole file is read, hands back the final fingerprint as text.
* `def sha1_file(path: Path | str) -> str:` — Defines a function exactly like the one above, but for an older, faster fingerprint format called SHA-1.
* `h = hashlib.sha1()` — Starts an empty SHA-1 fingerprint calculator.
* `with open(path, "rb") as f:` — Safely opens the file in "read binary" mode.
* `while chunk := f.read(_CHUNK):` — Reads the file one 1-Megabyte chunk at a time.
* `h.update(chunk)` — Adds the chunk into the SHA-1 calculator.
* `return h.hexdigest()` — Hands back the final SHA-1 fingerprint as text.

> Strict Data Checking

* `def _check(obj: Any, where: str = "$") -> None:` — Defines a hidden function that inspects data to ensure it is perfectly safe for strict JSON. The `where` variable tracks where we are in the data structure (starting at `$`, the root).
* `"""Reject anything whose JSON encoding is not unique: floats, NaN, non-string keys."""` — A comment explaining that decimal numbers and weird dictionary keys are banned because different computers format them differently.
* `if isinstance(obj, bool) or obj is None or isinstance(obj, (int, str)):` — Checks if the data is a safe, simple type: True/False, empty (`None`), a whole number, or text.
* `return` — If it is safe, stop checking and approve it (do nothing).
* `if isinstance(obj, float):` — Checks if the data is a decimal number (a float).
* `raise TypeError(f"float at {where}: canonical JSON is integers only")` — If it is a decimal, crash the program and print an error showing exactly where the decimal was found.
* `if isinstance(obj, dict):` — Checks if the data is a dictionary (a collection of key-value pairs).
* `for k, v in obj.items():` — Loops through every single key (`k`) and value (`v`) inside the dictionary.
* `if not isinstance(k, str):` — Checks if the dictionary key is anything other than text.
* `raise TypeError(f"non-string key {k!r} at {where}")` — If the key isn't text, crash the program with an error.
* `_check(v, f"{where}.{k}")` — Takes the dictionary value and sends it back through this exact same `_check` function to inspect it. This ensures nested data is checked all the way down.
* `return` — Once the whole dictionary is checked, stop.
* `if isinstance(obj, (list, tuple)):` — Checks if the data is a list or tuple (an ordered collection of items).
* `for i, v in enumerate(obj):` — Loops through the list, keeping track of the item's position number (`i`) and the item itself (`v`).
* `_check(v, f"{where}[{i}]")` — Sends each list item back through the `_check` function to inspect it, recording its position.
* `return` — Once the whole list is checked, stop.
* `raise TypeError(f"unsupported type {type(obj).__name__} at {where}")` — If the data isn't any of the allowed types above, crash the program and say the data type is unsupported.

> JSON Formatting (Canonical JSON)

* `def canonical_json(obj: Any) -> bytes:` — Defines a function that turns data into a highly strict, squished-down JSON byte format.
* `"""Compact form: stage keys, JSONL lines, journals."""` — A comment saying this squished format is meant for tight storage and logs.
* `_check(obj)` — Runs the data through our `_check` function above to make sure it contains no banned elements.
* `text = json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",", ":"))` — Converts the data into text JSON. It forces the keys into alphabetical order, allows foreign language characters, and removes all extra spaces.
* `return text.encode("utf-8")` — Converts that strict JSON text into raw computer bytes using the standard UTF-8 format.
* `def canonical_json_pretty(obj: Any) -> bytes:` — Defines a function that turns data into strict JSON, but formatted beautifully for human eyes.
* `"""Manifest form: sorted keys, indent 2, UTF-8, trailing newline."""` — A comment explaining this format is spaced out and easy to read.
* `_check(obj)` — Checks the data for banned elements again.
* `text = json.dumps(obj, sort_keys=True, ensure_ascii=False, indent=2, separators=(",", ": "))` — Converts the data into text JSON, sorts the keys, adds a 2-space indentation for neatness, and puts spaces after colons.
* `return (text + "\n").encode("utf-8")` — Adds an invisible "Enter" (new line) at the very end of the text, then converts it to bytes.

> Document Identity (Unique IDs)

* `def doc_uid(wiki: str, page_id: int) -> int:` — Defines a function that creates a unique ID number by combining a wiki name and a page number.
* `"""Little-endian u64 of the first 8 bytes of sha256(f"{wiki}:{page_id}")."""` — A comment explaining the math used to generate the ID.
* `digest = hashlib.sha256(f"{wiki}:{page_id}".encode()).digest()` — Mashes the wiki name and page ID together into text (like "enwiki:123"), converts it to bytes, and generates a raw SHA-256 fingerprint.
* `return int.from_bytes(digest[:8], "little")` — Chops off just the first 8 bytes of that fingerprint and converts them into a massive whole number.
* `def uid_hex(uid: int) -> str:` — Defines a function that takes that massive whole number ID and turns it into text.
* `"""JSON form of a doc_uid: hex of the integer, so string order equals numeric order."""` — Explains this makes the ID safe to save in JSON while making sure alphabetical order matches numerical order.
* `return f"{uid:016x}"` — Converts the number into a 16-character text string using letters and numbers, adding zeros to the front if it is too short.
* `def parse_uid_hex(text: str) -> int:` — Defines a function that takes the 16-character text ID and turns it back into a number.
* `if len(text) != 16:` — Checks if the text is exactly 16 characters long.
* `raise ValueError(f"doc_uid must be 16 hex digits, got {text!r}")` — If it isn't exactly 16 characters, crash the program with an error.
* `return int(text, 16)` — Converts the 16-character text string back into a standard whole number.
* `def text_sha256(text: str) -> str:` — Defines a simple function to quickly get the fingerprint of plain text.
* `return sha256_hex(text.encode("utf-8"))` — Converts the text into raw bytes and passes it to our very first function (`sha256_hex`) to get the fingerprint.

### Step 0.6: Fault injection

`src/wiki2shards/pipeline/faults.py`

| Code Line | Plain English Explanation |
| --- | --- |
| `"""Fault injection: W2S_FAULT="<stage>:<point>:<n>" kills the coordinator at the n-th hit.` | Opens a multi-line comment describing what this code does: it's a tool to intentionally crash a program by setting a specific rule. |
| `os._exit skips finally blocks, atexit handlers and buffered flushes, exactly like SIGKILL, so the` | Continues the comment, explaining that this tool forces a brutal, instant crash without giving the program any time to clean up its files or memory. |
| `crash tests see the same on-disk state a real kill would leave behind.` | Concludes the comment, noting that this helps developers test how the system reacts to sudden power cuts or forced closures. |
| `"""` | Closes the multi-line comment block. |
| `from __future__ import annotations` | A special instruction that helps developers write clearer, more modern code labels (type hints) in older versions of Python. |
| `import os` | Brings in Python's built-in operating system tools, which are needed to read computer settings and force the program to quit. |
| `FAULT_EXIT_CODE = 137` | Sets a specific error number (137) that the program will announce when it intentionally crashes itself. |
| `_hits: dict[tuple[str, str], int] = {}` | Creates an empty digital scoreboard (a dictionary) to keep a running tally of how many times different parts of the code are reached. |
| `def hit(stage: str, point: str) -> None:` | Creates a reusable block of code (a function) named `hit`. You give it a `stage` (a phase of the program) and a `point` (a specific spot). |
| `    spec = os.environ.get("W2S_FAULT")` | Looks at your computer's hidden settings (environment variables) to see if you gave it a crash instruction named "W2S_FAULT". |
| `    if not spec:` | Checks if that crash instruction is missing or empty. |
| `        return` | If there is no crash instruction, exit this block of code immediately and let the program continue running normally. |
| `    want_stage, want_point, n = spec.split(":")` | If there *is* an instruction, chop it into three pieces separated by colons: the exact stage, the exact point, and the number of times to hit it. |
| `    if (stage, point) != (want_stage, want_point):` | Checks if the program's current location matches the exact location where you told it to crash. |
| `        return` | If the locations don't match, exit this block of code and let the program continue normally. |
| `    count = _hits.get((stage, point), 0) + 1` | Checks the scoreboard to see how many times the program has visited this exact spot before, and adds 1 to that number. |
| `    _hits[(stage, point)] = count` | Updates the scoreboard with the new, higher count. |
| `    if count == int(n):` | Checks if the number of times we've visited this spot exactly matches the target crash number you set. |
| `        os._exit(FAULT_EXIT_CODE)` | If the numbers match, instantly and violently kill the program, spitting out the error number 137. |

### Step 0.7: The fixture dump generator

`scripts/make_fixture_dump.py`

Here is a plain-English breakdown of what this script does, section by section.

This script generates **fake Wikipedia database dumps** for testing software. Instead of downloading gigabytes of real Wikipedia data, developers can run this script to generate a small, predictable, "dummy" version of Wikipedia that always looks exactly the same every time it is generated.

> 1. Introduction and Setup

* **The Docstring (`""" ... """`)**: Explains what the script does. It creates files that perfectly mimic the structure of real Wikimedia dump files (XML compressed with bz2). It emphasizes that it uses no random number generators—everything is strictly calculated so the output never changes between computers.
* **Imports**: Loads built-in Python tools needed for the job: `argparse` (to read command-line commands), `bz2` (to compress files), `hashlib` (for generating predictable numbers/checksums), `json` (for saving test answers), `Path` (for handling file folders), and `escape/quoteattr` (to safely format text into XML).

> 2. The Raw Ingredients (Constants)

* `PAGES_PER_STREAM = 100`: Groups 100 wiki pages into a single compressed block, just like real Wikipedia dumps.
* `BASE_DATE`: A fake date used for the file names.
* `WORDS`: A giant list of simple English words. The script uses these to generate fake sentences and page titles.
* `COUNTS`: A dictionary deciding exactly how many pages of specific "weird" types to create (e.g., 20 redirects, 10 talk pages, 1 emoji page).
* `TOTAL_PAGES = 300`: The script will generate exactly 300 pages in total.

> 3. Generating Predictable "Random" Text

Instead of using Python's `random` module (which can act differently on different machines), the script uses math to pick words.

* **`_h` function**: Takes any input (like a page name), hashes it using SHA-256, and turns it into a permanent, unique number.
* **`words` function**: Uses that unique number to pick words from the `WORDS` list over and over to form a list of fake words.
* **`prose` function**: Takes those words, capitalizes the first letter, adds periods, and groups them into 5-sentence paragraphs to look like real article text.
* **`article_wikitext` function**: Glues the fake paragraphs together with real MediaWiki formatting (like `[[links]]`, `{{infoboxes}}`, `<ref>` tags, and tables) so the testing software has complex formatting to practice reading.

> 4. Building the Pages

* **`Page` class**: A simple blueprint to hold a page's data (its title, category, namespace ID, and text).
* **`build_pages` function**:
* Figures out how many standard "article" pages to make to hit exactly 300 pages.
* Loops through every category (redirects, empty pages, short pages, foreign languages, etc.) and generates the text for them using the functions above.
* Sorts them using the `_h` math function so they are thoroughly—but predictably—mixed up.
* If `variant 2` is requested (simulating a wiki update that happens later in time), it deletes 5 pages, edits a few, and adds 5 new ones.

> 5. Keeping IDs Stable

* **`page_ids` function**: Real Wikipedia pages have ID numbers that never change, even if other pages are deleted. This function calculates permanent ID numbers for every generated page.

> 6. Formatting as XML

* **`page_xml` function**: Takes a `Page` object and wraps it in MediaWiki XML tags (like `<page>`, `<title>`, `<text>`).
* **`header_xml` function**: Creates the standard XML header that sits at the very top of the dump file, declaring the site's name as "Fixturepedia".

> 7. Creating the "Answer Key"

* **`expected_answers` function**: Since this script generates test data, it also generates a "cheat sheet" (a JSON file). This file calculates exactly how many pages are valid, how many are empty, and how many are duplicates. The testing software will use this JSON file to grade itself and make sure it parsed the wiki dump correctly.

> 8. Compressing and Saving the Files

* **`make_dump` function**: This is where the actual files are built.
* It gathers all the pages and their IDs.
* It breaks the pages into chunks of 100.
* It converts them to XML, creates an index (so software knows exactly where in the file a specific page lives), and heavily compresses them into `.bz2` format.
* It creates the folder structure (`data/fixture_dump/fixturewiki/20260901/`).
* It writes the compressed XML file, the compressed index file, a `sha1sums.txt` file (to prove the files aren't corrupted), and the `fixture_expected.json` answer key to your hard drive.

> 9. The Execution Trigger

* **`main` function**: Sets up the command-line interface. When you type `python make_fixture_dump.py --out data`, this function reads your instructions and triggers `make_dump`.
* **`if __name__ == "__main__":`**: A standard Python rule that says, "If someone runs this script directly, start the `main()` function."

### Step 0.8: Shared test setup and the first tests

#### `tests/conftest.py`

`conftest.py` registers a deterministic Hypothesis profile and blocks network sockets. It also defines session-scoped fixtures that build the fixture dataset once per test run; those use helpers that arrive in later phases and are imported lazily.

```python
"""Shared test setup: Hypothesis profiles, a network guard, and session-scoped fixture builds."""

```

This is a comment at the very top of the file (a docstring) explaining what this file does: it sets up testing tools, blocks internet access during tests, and creates reusable test data.

```python
from __future__ import annotations

```

This allows Python to use modern type-hinting features (where you specify what type of data a variable is) without crashing on older Python versions.

```python
import atexit

```

Imports a built-in Python tool that lets you schedule tasks to run right before the program finishes and closes.

```python
import os

```

Imports a built-in tool to interact with your operating system, like reading environment variables or navigating folders.

```python
import shutil

```

Imports a built-in tool for managing files and folders, like deleting entire directories.

```python
import socket

```

Imports a built-in tool that Python uses to create network and internet connections.

```python
import tempfile

```

Imports a built-in tool used to create temporary files and folders that won't clutter your hard drive.

```python
from pathlib import Path

```

Imports a modern tool that makes it easier to work with file and folder paths across different operating systems (Windows, Mac, Linux).

```python
import pytest

```

Imports `pytest`, which is the main framework being used to write and run the tests.

```python
from hypothesis import HealthCheck, settings

```

Imports specific tools from `hypothesis`, a library that automatically generates test data to help find bugs.

```python
from hypothesis.configuration import set_hypothesis_home_dir

```

Imports a function to change the folder where `hypothesis` saves its background files and caches.

```python
REPO = Path(__file__).resolve().parents[1]

```

Calculates the exact folder path to the main project directory. It looks at the current file's path (`__file__`), gets the absolute path (`resolve`), and goes up two folder levels (`parents[1]`).

```python
os.chdir(REPO)  # configs use CWD-relative paths (tokenizer, language codes)

```

Changes the current working directory to the main project folder so that any files the tests need to read are found in the right place.

```python
# Hypothesis keeps caches (constants, unicode data) in ./.hypothesis even with database=None.
# Tests write only under temporary directories, so point its home at one for this session.

```

Comments explaining why the next few lines are necessary: to prevent `hypothesis` from leaving leftover cache files in the project folder.

```python
_HYPOTHESIS_HOME = tempfile.mkdtemp(prefix="w2s-hypothesis-")

```

Creates a new temporary folder on your computer starting with the name `w2s-hypothesis-` and saves its path to the variable `_HYPOTHESIS_HOME`.

```python
atexit.register(shutil.rmtree, _HYPOTHESIS_HOME, True)

```

Tells Python: "When the tests are completely finished, use `shutil.rmtree` to delete this temporary folder so it doesn't leave junk on the computer."

```python
set_hypothesis_home_dir(_HYPOTHESIS_HOME)

```

Tells `hypothesis` to use that temporary folder for its background files instead of the default project folder.

```python
# "ci" is deterministic and writes no example database (tests write only under tmp_path).

```

A comment explaining that the next block configures a testing profile for "CI" (Continuous Integration - the automated server that runs tests). It says it will be predictable and won't save files to the hard drive.

```python
settings.register_profile(

```

Starts the process of saving a specific configuration setup (a "profile") for `hypothesis`.

```python
    "ci",

```

Names this specific configuration profile "ci".

```python
    max_examples=60,

```

Tells `hypothesis` to generate and run a maximum of 60 random test cases per test.

```python
    deadline=None,

```

Turns off the time limit, meaning tests won't automatically fail if they run a little slowly.

```python
    derandomize=True,

```

Forces the randomly generated test data to be exactly the same every single time you run the tests, which makes debugging easier.

```python
    database=None,

```

Tells `hypothesis` not to save previously failed test cases to a database on the hard drive.

```python
    suppress_health_check=[HealthCheck.too_slow],

```

Tells `hypothesis` not to complain or throw a warning if generating the test data takes a long time.

```python
)

```

Closes the profile configuration block.

```python
settings.register_profile("explore", max_examples=500, deadline=None, database=None)

```

Creates a second configuration profile named "explore" that is more aggressive: it runs 500 examples instead of 60 to hunt for deeper bugs.

```python
settings.load_profile(os.environ.get("HYPOTHESIS_PROFILE", "ci"))

```

Activates a profile. It checks if your computer has an environment variable called `HYPOTHESIS_PROFILE` set (like "explore"). If not, it defaults to using the "ci" profile we just made.

```python
@pytest.fixture(autouse=True)

```

A special tag (decorator) that tells `pytest`: "The following function is a setup task, and you should run it automatically before every single test."

```python
def _no_network(monkeypatch: pytest.MonkeyPatch) -> None:

```

Defines a function named `_no_network` that takes a tool called `monkeypatch` (which lets you temporarily replace parts of Python). `-> None` means this function doesn't return a value.

```python
    """Tests never touch the network: refuse any internet socket connection."""

```

A comment explaining the function's purpose: to strictly block the tests from accessing the internet.

```python
    real_connect = socket.socket.connect

```

Saves a copy of Python's real, original function that connects to the internet, so it can still be used for local (non-internet) connections.

```python
    def guarded(self: socket.socket, address: object) -> None:

```

Defines a fake, custom connection function inside our setup function, which will act as a security guard.

```python
        if self.family in (socket.AF_INET, socket.AF_INET6):

```

Checks if the connection being attempted is an IPv4 or IPv6 internet connection.

```python
            raise RuntimeError(f"network access attempted in a test: {address!r}")

```

If it *is* an internet connection, this deliberately crashes the test with an error message showing what address it tried to connect to.

```python
        real_connect(self, address)  # type: ignore[arg-type]

```

If it is a safe local connection (like reading a local file socket), it goes ahead and uses the `real_connect` function we saved earlier.

```python
    monkeypatch.setattr(socket.socket, "connect", guarded)

```

Takes our fake `guarded` function and temporarily replaces Python's built-in internet connection tool with it for the duration of the tests.

```python
@pytest.fixture(scope="session")

```

A tag telling `pytest`: "The following setup function is expensive, so only run it once for the entire test session, and let all tests share the result."

```python
def dumps(tmp_path_factory: pytest.TempPathFactory) -> Path:

```

Defines a function named `dumps` that asks `pytest` for a tool to make temporary folders (`tmp_path_factory`). It promises to return a file `Path`.

```python
    """Fixture dumps: fixturewiki at two dates, plus a second wiki for two-source mixtures."""

```

A comment explaining that this function builds fake Wikipedia data dumps for testing purposes.

```python
    from helpers.build import make_dumps

```

Imports a custom helper function named `make_dumps` from another file in the project. (It imports it here instead of at the top of the file so it only loads if this function actually runs).

```python
    return make_dumps(tmp_path_factory.mktemp("dumps"))

```

Creates a temporary folder named "dumps", builds the fake data inside it using `make_dumps`, and hands that folder's path to any test that asks for it.

```python
@pytest.fixture(scope="session")

```

Another tag telling `pytest` to run the following setup function only once for the whole test session.

```python
def fixture_build(dumps: Path, tmp_path_factory: pytest.TempPathFactory):  # type: ignore[no-untyped-def]

```

Defines a function named `fixture_build` that requires the `dumps` folder we just made in the previous step, and the temporary folder tool.

```python
    from helpers.build import build_fixture

```

Imports another custom helper function called `build_fixture` from a local project file.

```python
    return build_fixture(tmp_path_factory.mktemp("build_a"), dumps)

```

Creates a new temporary folder named "build_a", processes the `dumps` data into a usable test environment, and returns it for tests to use.

```python
@pytest.fixture(scope="session")

```

Another tag telling `pytest` to run this setup function only once per session.

```python
def fixture_build_b(dumps: Path, tmp_path_factory: pytest.TempPathFactory):  # type: ignore[no-untyped-def]

```

Defines a function named `fixture_build_b` to create a second, slightly different test environment, also requiring the original `dumps` folder.

```python
    from helpers.build import build_fixture

```

Imports the `build_fixture` tool again.

```python
    return build_fixture(tmp_path_factory.mktemp("build_b"), dumps, wiki="fixture2wiki")

```

Creates a temporary folder named "build_b" and builds a second environment based on a different fake wiki named "fixture2wiki", making it available for tests that need two different setups at once.

#### `tests/unit/test_io_atomic.py`

This code tests a custom file-saving tool to ensure it saves files "atomically"—meaning the save is all-or-nothing. If a crash happens while saving, it guarantees you won't be left with a corrupted, half-written file.

Here is what every line does:

* `import subprocess`: Brings in a Python tool that allows this script to launch and run other programs in the background.
* `import sys`: Brings in a tool to interact with the Python system itself (used later to find the Python launcher).
* *(blank line)*
* `import pytest`: Brings in a popular testing framework used to verify that the code works as expected.
* *(blank line)*
* `from wiki2shards.io.atomic import atomic_open, atomic_write_bytes`: Loads two custom tools from the project's own code designed to safely write files without leaving them half-finished.
* *(blank line)*
* *(blank line)*
* `def test_atomic_write(tmp_path):`: Defines a new test function and gives it a temporary, disposable folder (`tmp_path`) to run tests inside safely.
* `    final = tmp_path / "out.bin"`: Creates a file path named "out.bin" inside that temporary folder.
* `    with pytest.raises(RuntimeError), atomic_open(final) as f:`: Tells the test to *expect* a `RuntimeError` crash to happen in the next block, and opens the file so we can start writing to it.
* `        f.write(b"partial")`: Writes the word "partial" into the file (the `b` means it is written as raw bytes, not standard text).
* `        raise RuntimeError("boom")`: Deliberately forces the program to crash with an error named "boom" right in the middle of writing.
* `    assert not final.exists()`: Checks to ensure the final "out.bin" file does not exist, proving the failed save didn't leave a half-written file behind.
* `    assert list(tmp_path.iterdir()) == []  # the tmp file is gone too`: Looks inside the temporary folder to guarantee no hidden leftover junk files were left behind either.
* *(blank line)*
* `    atomic_write_bytes(final, b"v1")`: Successfully writes the text "v1" into the file using the safe saving tool.
* `    with pytest.raises(RuntimeError), atomic_open(final) as f:`: Again, sets up a block that expects a crash, and opens the file to try and overwrite it.
* `        f.write(b"v2, half written")`: Starts writing new data over the old file.
* `        raise RuntimeError("boom")`: Deliberately forces another crash before the save can finish.
* `    assert final.read_bytes() == b"v1"  # the old content survives a failed rewrite`: Reads the file and verifies it still perfectly contains the original "v1" data, untouched by the failed overwrite attempt.
* *(blank line)*
* `    # A hard kill mid-write runs no cleanup at all; still nothing appears under the final name.`: A comment explaining that the next test will brutally kill the program without letting it clean up its mess.
* `    killed = tmp_path / "killed.bin"`: Creates a new file path named "killed.bin" in the temporary folder.
* `    code = (`: Begins a block of text that contains a miniature Python script.
* `        "import os, sys\nfrom wiki2shards.io.atomic import atomic_open\n"`: The mini-script loads necessary system tools and the custom safe-open tool.
* `        "with atomic_open(sys.argv[1]) as f:\n    f.write(b'x' * 4096)\n    f.flush()\n"`: The mini-script opens the file, writes 4,096 "x" characters, and forces the computer to physically push that data to the hard drive.
* `        "    os._exit(137)\n"`: The mini-script instantly and violently kills itself (using error code 137), completely bypassing any normal file-closing or cleanup steps.
* `    )`: Closes the block of text containing the mini-script.
* `    proc = subprocess.run([sys.executable, "-c", code, str(killed)])`: Uses the system to launch a completely separate background process to run that mini-script, pointing it at the "killed.bin" file path.
* `    assert proc.returncode == 137`: Verifies that the background process actually died violently with the 137 error code.
* `    assert not killed.exists()`: Checks to ensure that even though the program was killed brutally mid-save, the "killed.bin" file was never officially created, preventing data corruption.

#### `tests/unit/test_io_hashing.py`

Here is a line-by-line breakdown of what this Python test code is doing in plain English:

**`import pytest`**
Brings in a popular testing tool called `pytest`, which is used to check if the code is working correctly.

**`from wiki2shards.io.hashing import (`**
Starts importing specific tools (functions) from a file inside the project called `hashing.py`.

**`    canonical_json,`**
Imports a function that formats data into a standardized (canonical) JSON format so it always looks exactly the same.

**`    canonical_json_pretty,`**
Imports a function that formats data into a standardized JSON format, but spaced out so it's easy for humans to read.

**`    doc_uid,`**
Imports a function that creates a unique ID number for a document.

**`    parse_uid_hex,`**
Imports a function that translates a text-based (hexadecimal) ID back into a regular number.

**`    uid_hex,`**
Imports a function that translates a regular ID number into a text-based hexadecimal format (numbers 0-9 and letters a-f).

**`)`**
Closes the list of things being imported.

**`def test_canonical_json():`**
Defines a new test function meant to check if the JSON formatting tools work properly.

**`    a = {"b": [1, 2, {"z": None, "a": True}], "a": "é", "c": {"y": 1, "x": 2}}`**
Creates a bundle of data (a dictionary) named `a` containing mixed information like numbers, text with a special character (`é`), true/false values, and blank values (`None`).

**`    b = {"c": {"x": 2, "y": 1}, "a": "é", "b": [1, 2, {"a": True, "z": None}]}`**
Creates a second bundle of data named `b` with the exact same information as `a`, but written in a completely different order.

**`    assert canonical_json(a) == canonical_json(b)`**
Checks (asserts) that if you run both `a` and `b` through the formatting tool, the result is completely identical, proving the tool automatically sorts out scrambled orders.

**`    assert canonical_json(a) == '{"a":"é","b":[1,2,{"a":true,"z":null}],"c":{"x":2,"y":1}}'.encode()`**
Checks that the formatted version of `a` exactly matches this specific, correctly sorted text string that has been converted into computer bytes (which is what `.encode()` does).

**`    assert canonical_json_pretty(a).endswith(b"}\n")`**
Checks that the "pretty" human-readable version of the data ends properly with a closing bracket `}` and an invisible "new line" character `\n`.

**`    assert canonical_json_pretty(a) == canonical_json_pretty(b)`**
Checks that the "pretty" human-readable versions of `a` and `b` are also perfectly identical despite starting in different orders.

**`    for bad in (1.5, {"x": [0.0]}, {"x": float("nan")}, {1: "non-string key"}, {"s": {1, 2}}):`**
Starts a loop that goes through a list of invalid data types (like decimal numbers, "Not a Number" values, or keys that aren't text), temporarily calling each one `bad`.

**`        with pytest.raises(TypeError):`**
Tells the testing tool to expect the very next line of code to crash and throw a specific error called a `TypeError`.

**`            canonical_json(bad)`**
Tries to run the bad data through the formatting tool. If it successfully throws the expected error, the test passes. If it doesn't crash, the test fails.

**`def test_doc_uid_golden():`**
Defines a second test function to check if the unique document ID generators are working correctly.

**`    # Pinned values: a built-in hash() implementation would fail here in-process.`**
This is just a note for developers reading the code. It explains that they are testing against specific, hard-coded results because Python's default randomization would make the IDs change every time the test runs.

**`    assert uid_hex(doc_uid("enwiki", 12)) == "6b94dc5bf1a07c42"`**
Checks that generating a unique ID for document number 12 in the English Wiki ("enwiki") and converting it to text always results in exactly "6b94dc5bf1a07c42".

**`    assert uid_hex(doc_uid("simplewiki", 12)) == "0e055fda4b07921c"`**
Checks that doing the same thing for document number 12 in a different wiki ("simplewiki") results in a completely different, but expected, specific text string.

**`    assert parse_uid_hex(uid_hex(2**64 - 1)) == 2**64 - 1`**
Checks that if you take the maximum possible 64-bit number (`2**64 - 1`), convert it to text, and then immediately translate it back to a number, you get the exact same number you started with (proving the translation works perfectly in reverse).

### Validation

Here is the line-by-line explanation of what these commands do:

1. **`uv run python scripts/make_fixture_dump.py --out data/fixture_dump`**
This uses `uv` (a fast Python runner) to execute a script named `make_fixture_dump.py`. The `--out` part tells the script to save the files it generates into a folder called `data/fixture_dump`.
2. **`uv run python scripts/make_fixture_dump.py --out data/fixture_dump_again`**
This runs the exact same script again, but this time it saves the generated files into a different folder called `data/fixture_dump_again`.
3. **`diff -r data/fixture_dump data/fixture_dump_again && echo 'fixture dump: byte-identical'`**
This compares the contents of the two folders (`diff -r`) to see if there are any differences. The `&&` means "if they match perfectly, do the next thing." If they are exactly identical, it prints the message "fixture dump: byte-identical" to the screen. (This is testing to ensure the script produces reliable, identical results every time it runs).
4. **`(cd data/fixture_dump/fixturewiki/20260901 && shasum -a 256 *)`**
This temporarily goes into a specific folder (`cd ...`) and generates a unique digital fingerprint called a SHA-256 hash (`shasum -a 256`) for every file (`*`) in that folder. The parentheses ensure that once this is done, you are automatically put back in your original starting folder.
5. **`uv run pytest -q tests/unit/test_io_atomic.py tests/unit/test_io_hashing.py`**
This runs automated software tests on two specific files (`test_io_atomic.py` and `test_io_hashing.py`) using a Python testing tool called `pytest`. The `-q` stands for "quiet," meaning it will output a simplified, less cluttered summary of the test results.
6. **`uv run ruff check . && uv run ruff format --check . && uv run mypy src`**
This runs three different code-quality checks in a row, only moving to the next one if the previous one passes:
* `ruff check .` scans the code for bugs and bad practices (linting).
* `ruff format --check .` checks if the code is styled and spaced cleanly without actually altering it.
* `mypy src` checks the main code folder (`src`) to ensure all variables and data types are used correctly (type checking).

```sh
uv run python scripts/make_fixture_dump.py --out data/fixture_dump
data/fixture_dump/fixturewiki/20260901
```
---
```sh
uv run python scripts/make_fixture_dump.py --out data/fixture_dump_again
data/fixture_dump/fixturewiki/20260901
```
---
```sh
diff -r data/fixture_dump data/fixture_dump_again && echo 'fixture dump: byte-identical'
fixture dump: byte-identical
```
---
```sh
(cd data/fixture_dump/fixturewiki/20260901 && shasum -a 256 *)

2c11b79c7600e0060103ba5f19bb324234a3eaa4626fda030971a0e4a6148edc  fixture_expected.json
da6b5bdbe15a86f1600cef2b8b8118230e0d40f0cbb4ce5b57dd37424474834a  fixturewiki-20260901-pages-articles-multistream-index.txt.bz2
245358bf9f73289ed2f9266f00c47c68bf16eaef1773ea73a26a18c60983a77f  fixturewiki-20260901-pages-articles-multistream.xml.bz2
5ff7e35574c64508a57f348ca9a1bda75cb9e4d522fe85c26d2a65dec84223ca  fixturewiki-20260901-sha1sums.txt
```
---
```sh
uv run pytest -q tests/unit/test_io_atomic.py tests/unit/test_io_hashing.py
...                                                                                                                                                                                                                [100%]
3 passed in 0.04s
```
---
```sh
uv run ruff check . && uv run ruff format --check . && uv run mypy src
All checks passed!
19 files already formatted
Success: no issues found in 12 source files
```
---

## Phase 1: Contracts, then freeze (M1)

`src/wiki2shards/io/binfmt.py`

### Step 1.1: .bin/.idx, the dtype guard and shard checkers

```python
"""The .bin/.idx shard format (SPEC 1.4). Frozen at contracts-v1.

.bin: n_rows x row_len tokens, row-major, <u2 or <u4, no header, no trailing bytes.
.idx: 128-byte little-endian header, then row_seg_ptr <u8[n_rows+1], seg_doc_uid <u8[n_seg],
      seg_start <u4[n_seg], seg_len <u4[n_seg], seg_piece <u4[n_seg].
"""

```

This is a multi-line comment (docstring) explaining what this code does. It defines a specific way to store data in two files: a `.bin` file containing raw data tokens (words/pieces of text converted to numbers) and an `.idx` file containing an index (like a table of contents) to navigate the `.bin` file.

```python
from __future__ import annotations

```

Allows the code to use modern Python type hinting features before they are fully loaded by the interpreter.

```python
import struct
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from numpy.typing import NDArray

```

These lines import tools the code needs:

* `struct`: Translates Python values into raw computer bytes (and vice versa).
* `dataclass`: A shortcut for creating classes that primarily store data.
* `Path`: A tool for working with file paths (like `C:/folder/file.txt`).
* `numpy as np`: A powerful math and array library, abbreviated as `np`.
* `NDArray`: A type hint for Numpy arrays.

```python
IDX_MAGIC = b"W2S-IDX\0"
IDX_VERSION_MAJOR = 1
IDX_VERSION_MINOR = 0

```

Defines constants used to identify a valid `.idx` file. The "magic" string is a unique byte signature at the start of the file ensuring it's the correct file type. It also sets the format version to 1.0.

```python
IDX_STRUCT = struct.Struct("<8sHHBBBBIIQQQQ32s32s8s")
HEADER_SIZE = 128
assert IDX_STRUCT.size == HEADER_SIZE

```

Creates a blueprint (`IDX_STRUCT`) for exactly how the 128-byte header is laid out in memory (using specific byte sizes like `<8s` for an 8-byte string, `H` for 2-byte integers, `I` for 4-byte integers, etc.). It then strictly verifies that this blueprint equals exactly 128 bytes.

```python
PACK_MODES = {"concat": 0, "bfd": 1}
SPLITS = {"train": 0, "val": 1}

```

Creates lookup dictionaries for different settings: how data is packed (`concat` vs `bfd`) and whether the data is for AI training (`train`) or validation (`val`).

```python
U2 = np.dtype("<u2")
U4 = np.dtype("<u4")
U8 = np.dtype("<u8")

```

Sets up shortcuts for specific Numpy data types: 16-bit (`U2`), 32-bit (`U4`), and 64-bit (`U8`) unsigned integers. The `<` means "little-endian" (how the computer reads the bytes).

```python
class FormatError(Exception):
    """Base class: the bytes on disk do not match the format."""

class BadMagic(FormatError):
    pass

class UnsupportedVersion(FormatError):
    pass

class SizeMismatch(FormatError):
    pass

class DtypeRangeError(ValueError):
    """A token id does not fit the target dtype (astype would wrap silently)."""

```

Defines custom error messages so if the file is broken, the program can crash with a highly specific explanation (e.g., `BadMagic` if the file signature is wrong, `SizeMismatch` if the file is cut off).

```python
def dtype_for_vocab(vocab_size: int) -> np.dtype[np.unsignedinteger]:
    return U2 if vocab_size <= 65_536 else U4

```

A function that looks at the total number of unique words/tokens (`vocab_size`). If there are 65,536 or fewer, it uses 2 bytes (`U2`) to save space. Otherwise, it uses 4 bytes (`U4`).

```python
def checked_cast(
    ids: NDArray[np.integer], dtype: np.dtype[np.unsignedinteger]
) -> NDArray[np.unsignedinteger]:
    """Cast token ids, refusing values astype would wrap (70000 -> 4464 for <u2)."""
    if ids.size:
        lo, hi = int(ids.min()), int(ids.max())
        info = np.iinfo(dtype)
        if lo < 0 or hi > info.max:
            raise DtypeRangeError(f"token ids span [{lo}, {hi}], outside {dtype.str} range")
    return ids.astype(dtype)

```

A safety function. When converting data types, it checks the lowest (`lo`) and highest (`hi`) numbers in the array. If any number is too big to fit in the new data type, it raises an error instead of silently corrupting the data.

```python
@dataclass(frozen=True)
class IdxHeader:

```

Creates a data container for the 128-byte header. `frozen=True` means once this header is created, its values cannot be accidentally changed.

```python
    itemsize: int
    pack_mode: int
    split: int
    row_len: int
    shard_index: int
    n_rows: int
    n_segments: int
    n_pad_tokens: int
    first_global_row: int
    bin_sha256: bytes
    tokenizer_fp: bytes
    version_major: int = IDX_VERSION_MAJOR
    version_minor: int = IDX_VERSION_MINOR

```

Lists all the individual pieces of information stored in the header, such as how many rows exist (`n_rows`), the size of the data (`itemsize`), and digital fingerprints (`bin_sha256`) to verify the file hasn't been tampered with.

```python
    def pack(self) -> bytes:
        return IDX_STRUCT.pack(
            IDX_MAGIC,
            self.version_major,
            self.version_minor,
            self.itemsize,
            self.pack_mode,
            self.split,
            0,
            self.row_len,
            self.shard_index,
            self.n_rows,
            self.n_segments,
            self.n_pad_tokens,
            self.first_global_row,
            self.bin_sha256,
            self.tokenizer_fp,
            b"\0" * 8,
        )

```

A method that takes all the Python variables listed above and squashes them into exactly 128 bytes of raw computer data using the `IDX_STRUCT` blueprint defined earlier. The `b"\0" * 8` is empty padding at the end.

```python
    @classmethod
    def unpack(cls, raw: bytes) -> IdxHeader:
        if len(raw) < HEADER_SIZE:
            raise SizeMismatch(f".idx shorter than its {HEADER_SIZE}-byte header")

```

A method that does the exact opposite: it takes raw computer bytes and turns them back into a Python `IdxHeader` object. First, it ensures the raw data is at least 128 bytes long.

```python
        (
            magic, vmaj, vmin, itemsize, pack_mode, split, _r, row_len,
            shard_index, n_rows, n_seg, n_pad, first_row, bin_sha, tok_fp, _r2,
        ) = IDX_STRUCT.unpack(raw[:HEADER_SIZE])

```

Slices out the first 128 bytes and extracts all the individual variables based on the blueprint.

```python
        if magic != IDX_MAGIC:
            raise BadMagic(f"bad .idx magic {magic!r}")
        if vmaj != IDX_VERSION_MAJOR:
            raise UnsupportedVersion(f".idx major version {vmaj}, reader supports {IDX_VERSION_MAJOR}")
        if itemsize not in (2, 4) or pack_mode not in (0, 1) or split not in (0, 1):
            raise FormatError(f"bad header enum: itemsize={itemsize} mode={pack_mode} split={split}")

```

Performs safety checks on the unpacked data: ensures the file signature matches, the version is supported, and the format settings are valid (e.g., itemsize must be 2 or 4).

```python
        return cls(itemsize, pack_mode, split, row_len, shard_index, n_rows, n_seg, n_pad, first_row, bin_sha, tok_fp, vmaj, vmin)

```

Returns the fully built `IdxHeader` object populated with the extracted values.

```python
@dataclass(frozen=True)
class Segments:
    row_seg_ptr: NDArray[np.uint64]
    doc_uid: NDArray[np.uint64]
    start: NDArray[np.uint32]
    length: NDArray[np.uint32]
    piece: NDArray[np.uint32]

```

Creates another data container that holds Numpy arrays representing "segments" (chunks of text). It tracks where chunks start, how long they are, and which document they belong to.

```python
    @property
    def n_rows(self) -> int:
        return int(self.row_seg_ptr.shape[0]) - 1

    @property
    def n_segments(self) -> int:
        return int(self.doc_uid.shape[0])

```

Helper properties that calculate the total number of rows and segments by checking the length of the arrays.

```python
def _counts(row_seg_ptr: NDArray[np.uint64]) -> NDArray[np.int64]:
    """Segments per row, from the CSR pointer."""
    ptr = row_seg_ptr.astype(np.int64)
    return ptr[1:] - ptr[:-1]

```

A math trick that takes an array of pointers (which show where things begin) and subtracts each pointer from the next one to figure out how many items exist in each row.

```python
def idx_size(n_rows: int, n_segments: int) -> int:
    return HEADER_SIZE + 8 * (n_rows + 1) + 20 * n_segments

```

A math formula calculating exactly how many bytes the `.idx` file should be, based on the number of rows and segments.

```python
def encode_idx(header: IdxHeader, seg: Segments) -> bytes:
    if seg.n_rows != header.n_rows or seg.n_segments != header.n_segments:
        raise ValueError("header counts disagree with the segment arrays")

```

A function to convert the entire index (header + segments) into raw bytes. It first checks that the header's counts match the actual data arrays.

```python
    return b"".join([
        header.pack(),
        seg.row_seg_ptr.astype(U8).tobytes(),
        seg.doc_uid.astype(U8).tobytes(),
        seg.start.astype(U4).tobytes(),
        seg.length.astype(U4).tobytes(),
        seg.piece.astype(U4).tobytes(),
    ])

```

Combines the packed header and converts all the Numpy arrays into raw byte strings, squishing them all together into one long string of bytes to be saved to disk.

```python
def decode_idx(raw: bytes) -> tuple[IdxHeader, Segments]:
    header = IdxHeader.unpack(raw)
    want = idx_size(header.n_rows, header.n_segments)
    if len(raw) != want:
        raise SizeMismatch(f".idx is {len(raw)} bytes, header implies {want}")

```

A function to convert a raw byte string back into a Header and Segments. It unpacks the header, calculates how long the file *should* be, and throws an error if the actual file size is different.

```python
    r, s = header.n_rows, header.n_segments
    off = HEADER_SIZE

```

Sets up variables `r` (rows) and `s` (segments), and creates an offset counter (`off`) starting at 128 (skipping the header).

```python
    def take(dtype: np.dtype[np.unsignedinteger], count: int) -> NDArray[np.unsignedinteger]:
        nonlocal off
        arr = np.frombuffer(raw, dtype=dtype, count=count, offset=off)
        off += dtype.itemsize * count
        return arr

```

A helper function that bites off chunks of the raw byte string. It reads `count` amount of numbers of type `dtype`, moving the `off` (offset) tracker forward so the next read starts in the correct place.

```python
    seg = Segments(take(U8, r + 1), take(U8, s), take(U4, s), take(U4, s), take(U4, s))
    return header, seg

```

Uses the `take` function five times to extract all the Numpy arrays, packages them into a `Segments` object, and returns both the header and the segments.

```python
def read_idx(path: Path | str) -> tuple[IdxHeader, Segments]:
    return decode_idx(Path(path).read_bytes())

```

A convenience function that takes a file path, opens it, reads all the bytes, and passes them to `decode_idx`.

```python
def read_idx_header(path: Path | str) -> IdxHeader:
    with open(path, "rb") as f:
        return IdxHeader.unpack(f.read(HEADER_SIZE))

```

Another convenience function that opens a file, but *only* reads the first 128 bytes to grab the header, saving memory if you don't need the massive segment arrays yet.

```python
def open_bin(
    path: Path | str, itemsize: int, n_rows: int, row_len: int
) -> NDArray[np.unsignedinteger]:
    dtype = U2 if itemsize == 2 else U4
    if n_rows == 0:
        return np.zeros((0, row_len), dtype=dtype)
    return np.memmap(path, dtype=dtype, mode="r", shape=(n_rows, row_len))

```

A function to read the `.bin` data file. Instead of loading potentially gigabytes of data into RAM at once, it uses `np.memmap`. This creates a virtual Numpy array that reads data directly off the hard drive only when requested. (If the file has 0 rows, it just returns an empty array).

```python
def check_segments(
    seg: Segments,
    *,
    row_len: int,
    pack_mode: int,
    allow_partial_last_row: bool = False,
) -> list[str]:
    """Structural invariants of SPEC 1.4 that need only the .idx arrays."""
    errors: list[str] = []

```

A massive validation function that checks the `Segments` arrays for mathematical and structural errors, building a list of error messages.

```python
    ptr = seg.row_seg_ptr.astype(np.int64)
    n_rows, n_seg = seg.n_rows, seg.n_segments
    if ptr[0] != 0 or ptr[-1] != n_seg:
        return [f"row_seg_ptr must run from 0 to {n_seg}, got {ptr[0]}..{ptr[-1]}"]

```

Checks that the pointers indicating where rows begin start perfectly at 0 and end perfectly at the total number of segments.

```python
    counts = ptr[1:] - ptr[:-1]
    if n_rows and counts.min() < 1:
        return ["row_seg_ptr is not strictly increasing (a row without segments)"]
    if n_seg == 0:
        return errors

```

Calculates segments per row. Checks if any row is totally empty (has 0 segments). If the whole file has 0 segments, it stops checking here and returns no errors.

```python
    start = seg.start.astype(np.int64)
    length = seg.length.astype(np.int64)
    if length.min() < 1:
        errors.append("segment with zero length")

```

Ensures every segment contains at least 1 token (no useless zero-length segments).

```python
    row_of = np.repeat(np.arange(n_rows), counts)
    first = ptr[:-1]
    if np.any(start[first] != 0):
        errors.append("a row's first segment does not start at position 0")

```

Figures out which row every segment belongs to. It then verifies that the very first segment in any row always starts at position index 0.

```python
    contiguous = np.ones(n_seg, dtype=bool)
    contiguous[first] = False
    prev_end = np.concatenate([[0], start[:-1] + length[:-1]])
    if np.any(start[contiguous] != prev_end[contiguous]):
        errors.append("segments within a row are not sorted and contiguous")

```

Checks that segments within a row connect perfectly back-to-back without gaps or overlapping (e.g., if segment 1 ends at position 5, segment 2 must start at position 5).

```python
    used = np.bincount(row_of, weights=length, minlength=n_rows).astype(np.int64)
    if np.any(used > row_len):
        errors.append("a row holds more than row_len tokens")

```

Adds up the length of all segments in a row and checks that no row overflows the maximum allowed width (`row_len`).

```python
    if pack_mode == PACK_MODES["concat"]:
        full = used == row_len
        if allow_partial_last_row and n_rows:
            full[-1] = True
        if not np.all(full):
            errors.append("concat row whose segments do not sum to row_len")
    return errors

```

If the file is using "concat" mode (where data is tightly packed together), it verifies that every single row is filled exactly to its maximum capacity. It can optionally forgive the very last row for being partially empty. It finally returns the list of any errors found.

```python
def check_tokens(
    tokens: NDArray[np.unsignedinteger],
    seg: Segments,
    *,
    vocab_size: int,
    pad_id: int,
) -> list[str]:
    """Token-level invariants: range, and pad_id after each row's last segment."""
    errors: list[str] = []

```

A second validation function that checks the actual raw tokens in the `.bin` file against the index rules.

```python
    if tokens.size and int(tokens.max()) >= vocab_size:
        errors.append(f"token id {int(tokens.max())} >= vocab_size {vocab_size}")

```

Ensures that no raw token number in the data is higher than the maximum allowed vocabulary size (e.g., if your dictionary has 50,000 words, you can't have a token ID of 50,001).

```python
    counts = _counts(seg.row_seg_ptr)
    row_of = np.repeat(np.arange(seg.n_rows), counts)
    used = np.bincount(row_of, weights=seg.length.astype(np.int64), minlength=seg.n_rows)

```

Recalculates how much space in each row is actually occupied by valid data segments.

```python
    row_len = tokens.shape[1] if tokens.ndim == 2 else 0
    pad_mask = np.arange(row_len)[None, :] >= used.astype(np.int64)[:, None]
    if np.any(tokens[pad_mask] != pad_id):
        errors.append("a padding position holds something other than pad_id")
    return errors

```

Calculates exactly which spots in the file are "empty" space at the ends of rows. It then verifies that every single empty spot contains the official "padding" token (e.g., a specific number like 0), and returns the list of errors.

### Step 1.2: The manifest

`src/wiki2shards/io/manifest.py`

This script defines the structure and rules for reading, writing, and validating a dataset's metadata file (called a "manifest"). Here is the line-by-line breakdown:

### Documentation and Imports

```python
"""dataset.manifest: schema, reader, writer (SPEC 1.4). Frozen at contracts-v1.

The manifest is canonical JSON written last; its presence commits the dataset and the SHA-256 of
its bytes is the dataset id (never stored inside it).
"""

```

* **Lines 1-5:** A multi-line comment (docstring) explaining what this file does. It states this is version 1.4 of the specification, the manifest is written in a strict JSON format, and the unique ID of the dataset is generated by hashing this file's contents.

```python
from __future__ import annotations

```

* **Line 7:** Allows Python to read type hints (like `list[SourceFile]`) without causing errors if the referenced types are defined later in the code.

```python
import json
from pathlib import Path
from typing import Any, Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError

from wiki2shards.io.atomic import atomic_write_bytes
from wiki2shards.io.hashing import canonical_json_pretty, sha256_hex

```

* **Line 9:** Imports the standard library tool for reading and writing JSON data.
* **Line 10:** Imports `Path` to easily handle file and folder paths.
* **Line 11:** Imports `Any` (allowing any data type) and `Literal` (restricting a value to specific exact choices) for type hinting.
* **Line 13:** Imports tools from the `pydantic` library, which is used to define strict data structures and validate that incoming data matches those structures.
* **Line 15:** Imports a custom function to safely write files so they don't get corrupted if the process is interrupted.
* **Line 16:** Imports custom functions to format JSON strictly consistently (`canonical_json_pretty`) and to calculate file hashes (`sha256_hex`).

### Constants and Error Handling

```python
MANIFEST_NAME = "dataset.manifest"
FORMAT = "wiki2shards"
FORMAT_VERSION = (1, 0)

```

* **Line 18:** Sets the standard filename for the manifest.
* **Line 19:** Sets the expected format name.
* **Line 20:** Sets the current version of the format as a tuple (Major 1, Minor 0).

```python
class ManifestError(Exception):
    pass

```

* **Lines 23-24:** Creates a custom error type specifically for manifest-related problems. `pass` means it does nothing special beyond acting as a standard error.

### Base Model Definition

```python
class _Model(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True, frozen=True)

```

* **Line 27:** Creates a foundational data model class based on Pydantic's `BaseModel`.
* **Line 28:** Configures all models using this base to reject any unlisted fields (`extra="forbid"`), strictly enforce data types (`strict=True`), and prevent changes after creation (`frozen=True`).

### Data Structure Definitions (Schemas)

```python
class BuildInfo(_Model):
    wiki2shards_version: str
    git_commit: str
    lockfile_sha256: str | None
    python: str
    unicode_version: str
    config_sha256: str

```

* **Lines 31-37:** Defines the `BuildInfo` section of the manifest. It requires strings detailing the software version, the exact Git code commit, the Python version, the text encoding version, and the settings hash. The lockfile hash can be a string or `None`.

```python
class SourceFile(_Model):
    name: str
    bytes: int
    sha1: str

```

* **Lines 40-43:** Defines a single downloaded source file, requiring its filename, size in bytes, and its SHA-1 hash for security.

```python
class SourceInfo(_Model):
    wiki: str
    dump_date: str
    files: list[SourceFile]

```

* **Lines 46-49:** Defines the `SourceInfo` section, tracking which Wikipedia version was used, the date of the dump, and a list of the `SourceFile` objects defined above.

```python
class TokenizerInfo(_Model):
    spec: str
    fingerprint: str
    vocab_size: int
    eos_id: int
    pad_id: int
    dtype: Literal["<u2", "<u4"]

```

* **Lines 52-58:** Defines settings for how text was chopped into tokens. It tracks the exact rules (`spec`, `fingerprint`), the total number of unique tokens (`vocab_size`), special token IDs (end-of-sequence and padding), and limits the data type (`dtype`) to specifically 16-bit or 32-bit unsigned integers.

```python
class PackingInfo(_Model):
    mode: Literal["concat", "bfd"]
    seq_len: int
    row_len: int
    doc_order: Literal["doc_uid_asc"]
    last_row: Literal["drop", "pad"]

```

* **Lines 61-66:** Defines how data was packed together. It forces `mode` to be one of two choices, sets integer lengths, forces document order to be ascending, and dictates whether the leftover last row should be deleted or padded with empty space.

```python
class SelectionInfo(_Model):
    policy: str
    soft_filters: list[str]
    dedup: str
    val_per_10k: int

```

* **Lines 69-73:** Details how data was filtered, listing the policy name, a list of applied filter names, the deduplication method used, and how many items per 10,000 are held back for testing/validation.

```python
class ShardEntry(_Model):
    index: int
    n_rows: int
    bin: str
    bin_bytes: int
    bin_sha256: str
    idx: str
    idx_bytes: int
    idx_sha256: str

```

* **Lines 76-84:** Defines a single "shard" (a broken-up piece of the dataset). It tracks the shard's sequence number, row count, file name for the raw data (`bin`), its size and hash, along with the file name, size, and hash for its corresponding index file (`idx`).

```python
class SplitInfo(_Model):
    n_rows: int
    n_tokens: int
    n_pad_tokens: int
    n_docs: int
    n_tokens_dropped_tail: int
    shards: list[ShardEntry]

```

* **Lines 87-93:** Defines a dataset split (like "train" or "validation"). It tallies up total rows, tokens, blank tokens, documents, discarded tokens, and holds a list of all the `ShardEntry` pieces that belong to this split.

```python
class SideFile(_Model):
    name: str
    bytes: int
    sha256: str

```

* **Lines 96-99:** Defines any extra supplementary files (like configurations), requiring a name, size, and hash.

### The Main Manifest Model

```python
class Manifest(_Model):
    format: Literal["wiki2shards"]
    format_version: list[int] = Field(min_length=2, max_length=2)
    name: str
    build: BuildInfo
    source: SourceInfo
    tokenizer: TokenizerInfo
    packing: PackingInfo
    selection: SelectionInfo
    splits: dict[Literal["train", "val"], SplitInfo]
    side_files: list[SideFile]

```

* **Line 102:** Starts the main `Manifest` structure that combines all the previous pieces.
* **Line 103:** Requires the format name to be exactly "wiki2shards".
* **Line 104:** Requires the version to be a list of exactly two numbers (e.g., `[1, 0]`).
* **Lines 105-112:** Embeds all the previously defined models (Build, Source, Tokenizer, etc.) as required sections. `splits` is required to be a dictionary with specific keys ("train" and "val"), mapping to the `SplitInfo` details.

```python
    def to_bytes(self) -> bytes:
        return canonical_json_pretty(self.model_dump(mode="python"))

```

* **Line 114:** Defines a function to convert the populated manifest model into raw bytes.
* **Line 115:** Extracts the data back into standard Python format, formats it into strict, pretty-printed JSON, and returns it.

### Helper and Processing Functions

```python
def _drop(data: Any, loc: tuple[int | str, ...]) -> None:
    for key in loc[:-1]:
        data = data[key]
    del data[loc[-1]]

```

* **Line 118:** Defines a helper function to delete a specific piece of data buried inside nested dictionaries.
* **Lines 119-120:** Loops through the path to find the exact nested location, stopping just before the final item.
* **Line 121:** Deletes the final item at that specific location.

```python
def parse_manifest(raw: bytes) -> Manifest:
    """Validate manifest bytes. Unknown keys are errors at minor 0 and ignored at higher minors."""

```

* **Lines 124-125:** Starts a function to take raw bytes and turn them into a valid `Manifest` object, with a comment explaining that newer file versions will ignore unrecognized fields rather than crashing.

```python
    try:
        data = json.loads(raw.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as exc:
        raise ManifestError(f"manifest is not valid UTF-8 JSON: {exc}") from exc

```

* **Lines 126-129:** Attempts to decode the raw bytes as standard text and read it as JSON. If it fails (due to bad text encoding or broken JSON syntax), it stops the program and raises a specific `ManifestError`.

```python
    if not isinstance(data, dict) or data.get("format") != FORMAT:
        raise ManifestError("not a wiki2shards manifest")

```

* **Lines 130-131:** Checks if the loaded JSON is a standard dictionary (object) and if its "format" key matches "wiki2shards". If not, it throws an error.

```python
    version = data.get("format_version")
    if not (isinstance(version, list) and len(version) == 2 and version[0] == FORMAT_VERSION[0]):
        raise ManifestError(f"unsupported format_version {version!r}")

```

* **Lines 132-134:** Extracts the version number. It verifies that the version is a 2-item list and that the major version number (the first number) matches the software's expected major version. If it's a mismatch, it throws an error.

```python
    try:
        return Manifest.model_validate(data)

```

* **Lines 135-136:** Tries to pass the raw dictionary into the strict `Manifest` Pydantic model. If it perfectly matches, it returns the validated object.

```python
    except ValidationError as exc:
        extras = [e["loc"] for e in exc.errors() if e["type"] == "extra_forbidden"]

```

* **Lines 137-138:** If validation fails, it catches the error. It checks if the failure was caused specifically by "extra" unknown fields being in the file.

```python
        if version[1] > FORMAT_VERSION[1] and len(extras) == len(exc.errors()):
            for loc in sorted(extras, key=len, reverse=True):
                _drop(data, loc)
            return Manifest.model_validate(data)

```

* **Lines 139-142:** If the file's minor version is newer than the software's, AND the *only* errors were these extra unexpected fields, it steps backwards through the file to delete those unknown extra fields, then tries validating it again.

```python
        raise ManifestError(f"manifest schema error: {exc}") from exc

```

* **Line 143:** If validation failed for any other reason (like missing required data or wrong data types), it throws a hard error.

```python
def read_manifest(dataset_dir: Path | str) -> tuple[Manifest, str]:
    """Return the manifest and the dataset id (sha256 of its bytes)."""
    raw = (Path(dataset_dir) / MANIFEST_NAME).read_bytes()

```

* **Lines 146-148:** Starts a function to read the manifest from a folder path. It combines the folder path with the filename and reads the raw file bytes from the hard drive.

```python
    manifest = parse_manifest(raw)

```

* **Line 149:** Passes those raw bytes to the `parse_manifest` function to validate and structure the data.

```python
    if manifest.format_version == list(FORMAT_VERSION) and manifest.to_bytes() != raw:
        raise ManifestError("manifest bytes are not in canonical form")

```

* **Lines 150-151:** Checks if the manifest is the exact same version as the software. If it is, but the raw file bytes don't exactly match the strictly formatted canonical bytes, it throws an error (meaning someone edited the JSON sloppily by hand).

```python
    return manifest, sha256_hex(raw)

```

* **Line 152:** Returns the final structured manifest object alongside the cryptographic hash (dataset ID) of the raw bytes.

```python
def write_manifest(dataset_dir: Path | str, manifest: Manifest) -> str:
    raw = manifest.to_bytes()
    atomic_write_bytes(Path(dataset_dir) / MANIFEST_NAME, raw)
    return sha256_hex(raw)

```

* **Lines 155-158:** Starts a function to save a manifest to a folder. It converts the manifest object back into raw strictly-formatted bytes. It then safely writes those bytes to the `dataset.manifest` file in the given folder, and finally returns the cryptographic hash of what it just wrote.

### Step 1.3: Inter-stage records

`src/wiki2shards/io/records.py`

```python
"""JSONL records passed between stages: extracted pages and cleaned documents.

```

This is the start of a multi-line comment (called a docstring) that explains the purpose of this file: it handles "JSONL" (JSON Lines) data that is being passed between different steps of a program.

```python
Lines are canonical JSON. Readers split on b"\\n" only: raw U+2028 is legal inside a JSON string,

```

This continues the comment, explaining a technical detail: each line is perfectly formatted JSON data. It warns that when reading the file, the computer should only separate lines using a specific "newline" character (`\n`), because other invisible spacing characters might exist inside the text and shouldn't accidentally break the line in half.

```python
and str.splitlines() would break a record there.

```

This finishes the comment, noting that Python's standard tool for splitting lines (`str.splitlines()`) would make a mistake by splitting on those invisible spacing characters.

```python
"""

```

This closes the multi-line comment.

```python
from __future__ import annotations

```

This is a special Python instruction that makes it easier to write "type hints" (labels that tell the programmer what kind of data to expect, like text or numbers) without causing errors in older versions of Python.

```python
import json

```

This brings in Python's built-in tool for reading and writing JSON (a popular, easy-to-read format for storing data).

```python
from collections.abc import Iterable, Iterator, Mapping

```

This imports special labels used to describe different types of data collections, such as things you can loop over (`Iterable`, `Iterator`) or things that link keys to values like a dictionary (`Mapping`).

```python
from pathlib import Path

```

This brings in a tool called `Path` that makes it much easier to work with file names and folders on your computer.

```python
from typing import Any, TypedDict

```

This brings in tools to help label data: `Any` means "this could be any type of data," and `TypedDict` lets the programmer create a strict blueprint for a dictionary.

```python
from wiki2shards.io.atomic import atomic_open

```

This imports a custom function called `atomic_open` from another part of this project. It is used to safely open and save files so that if the computer crashes halfway through saving, the file doesn't get corrupted.

```python
from wiki2shards.io.hashing import canonical_json

```

This imports another custom function, `canonical_json`, which converts data into a very strict, standardized JSON format so it always looks exactly the same.

```python
class ExtractedPage(TypedDict):

```

This creates a blueprint (a `TypedDict`) named `ExtractedPage`. It defines exactly what information must be included whenever the program handles a page extracted from a wiki.

```python
    page_id: int

```

The extracted page must have a `page_id`, which must be a whole number (an integer).

```python
    rev_id: int

```

The extracted page must have a `rev_id` (revision ID), which must also be a whole number.

```python
    title: str

```

The extracted page must have a `title`, which must be text (a string).

```python
    wikitext: str

```

The extracted page must have `wikitext` (the raw text from the wiki), which must be text.

```python
class CleanedDoc(TypedDict):

```

This creates another blueprint named `CleanedDoc` for documents that have been processed and cleaned up.

```python
    doc_uid: str  # 16 hex digits, see hashing.uid_hex

```

The cleaned document needs a unique ID (`doc_uid`) made of text. The comment next to it notes this will be a 16-character code.

```python
    page_id: int

```

It must have a `page_id` as a whole number.

```python
    rev_id: int

```

It must have a `rev_id` as a whole number.

```python
    title: str

```

It must have a `title` made of text.

```python
    text: str

```

It must have the cleaned `text` of the document, made of text.

```python
    n_chars: int

```

It must have a count of the characters (`n_chars`), which is a whole number.

```python
    text_sha256: str

```

It must have a special digital fingerprint of the text (`text_sha256`) to prove it hasn't been tampered with, stored as text.

```python
    verdicts: dict[str, bool]

```

It must have `verdicts` (like a checklist of rules the document passed or failed), which is a dictionary linking text names to True/False values (booleans).

```python
def write_jsonl(path: Path | str, records: Iterable[Mapping[str, Any]]) -> None:

```

This defines a new function named `write_jsonl`. It takes two inputs: a file path (where to save the file) and a list of records (the data to save). The `-> None` part just means this function performs an action but doesn't hand any data back to you when it finishes.

```python
    with atomic_open(path) as f:

```

This safely opens the file at the given path so it can be written to, and temporarily names the open file `f`.

```python
        for rec in records:

```

This starts a loop that will look at every single record (piece of data) in the list of records provided.

```python
            f.write(canonical_json(dict(rec)) + b"\n")

```

For each record, this line turns it into a standard dictionary, converts it to the strict JSON format, adds a "newline" character (like hitting the Enter key) to the end, and writes that whole line into the file.

```python
def iter_jsonl(path: Path | str) -> Iterator[dict[str, Any]]:

```

This defines a new function named `iter_jsonl`. It takes a file path as input and is designed to hand back one dictionary record at a time (an Iterator) so the computer doesn't get overwhelmed reading a massive file all at once.

```python
    data = Path(path).read_bytes()

```

This reads the entire contents of the file at the given path into the computer's memory as raw, basic computer bytes.

```python
    for line in data.split(b"\n"):

```

This chops that raw data into separate pieces every time it sees a "newline" character (`\n`), and starts a loop to look at each individual line.

```python
        if line:

```

This checks to make sure the line isn't completely empty (which often happens at the very end of a file).

```python
            yield json.loads(line)

```

If the line has data, this converts the JSON text back into a usable Python dictionary and pauses to hand it back to whatever part of the program asked for it, before looping around to the next line.

### Step 1.4: Config models

`src/wiki2shards/config.py`

Here is the line-by-line explanation of the provided Python code:

### Module Docstring and Imports

`"""Build and mixture configs: pydantic models loaded from YAML. Frozen at contracts-v1.`
Starts a multi-line comment explaining this file handles configuration models loaded from YAML files, locked to a specific version.

`extra="forbid" turns typos into errors; strict=True stops YAML 1.1 coercions (an unquoted`
Explains that strict settings are used so accidental typos are rejected instead of ignored.

`20260901 is an int, an unquoted \`no` is False) from being silently accepted.`
Gives examples of how strict settings prevent strings from accidentally being converted to numbers or booleans.

`"""`
Ends the multi-line comment.

`from __future__ import annotations`
Allows the code to use modern Python type hints (like referring to a class before it is fully defined).

`from pathlib import Path`
Imports the `Path` tool to make working with file and folder paths easier.

`from typing import Any, Literal`
Imports `Any` (allowing any data type) and `Literal` (restricting a value to specific exact choices) for type hinting.

`import yaml`
Imports the `yaml` library to read YAML configuration files.

`from pydantic import BaseModel, ConfigDict, Field, model_validator`
Imports tools from the `pydantic` library used to build, validate, and restrict the configuration data models.

`from wiki2shards.io.hashing import canonical_json, sha256_file, sha256_hex`
Imports custom hashing functions from another part of the project to help generate unique fingerprints for files and data.

`MIB = 1 << 20`
Defines a constant named `MIB` equal to 1,048,576 (which is 1 Megabyte in binary bytes), calculated by shifting the number 1 left by 20 bits.

### Base Classes and Exceptions

`class ConfigError(Exception):`
Creates a custom error type called `ConfigError` specifically for when something goes wrong with the configuration.

`    pass`
Means this custom error doesn't need any special behavior beyond acting as a standard error.

`class _Strict(BaseModel):`
Creates a base configuration model called `_Strict` that inherits from Pydantic's `BaseModel`.

`    model_config = ConfigDict(extra="forbid", strict=True, frozen=True)`
Configures this base model to reject unknown fields (`extra="forbid"`), strictly enforce data types (`strict=True`), and prevent changes after creation (`frozen=True`).

### Configuration Data Models

`class SourceConfig(_Strict):`
Defines the `SourceConfig` model (inheriting the strict rules) to handle where the Wikipedia data comes from.

`    wiki: str = Field(pattern=r"^[a-z0-9_]+$")`
Requires a `wiki` string that must contain only lowercase letters, numbers, and underscores.

`    date: str = Field(pattern=r"^\d{8}$")`
Requires a `date` string that must be exactly 8 digits long (e.g., "20261010").

`    # Base URL (https://...), or a local directory laid out as <mirror>/<wiki>/<date>/`
A comment explaining what the `mirror` field should look like.

`    mirror: str = "[https://dumps.wikimedia.org](https://dumps.wikimedia.org)"`
Defines `mirror` with a default web address where Wikipedia dumps are downloaded from.

`class ExtractConfig(_Strict):`
Defines the `ExtractConfig` model for settings related to extracting data.

`    unit_mb: int = Field(default=32, ge=1)`
Defines `unit_mb` as an integer with a default of 32, which must be greater than or equal to (`ge`) 1.

`    unit_bytes: int | None = Field(default=None, ge=1)  # overrides unit_mb; for tiny fixtures`
Defines an optional `unit_bytes` that defaults to nothing, but must be at least 1 if provided (used for testing small files).

`    @property`
Marks the next function as a property, allowing it to be accessed like an attribute (e.g., `config.target_bytes`).

`    def target_bytes(self) -> int:`
Defines the `target_bytes` property which calculates and returns an integer.

`        return self.unit_bytes if self.unit_bytes is not None else self.unit_mb * MIB`
Returns `unit_bytes` if it was provided; otherwise, it calculates the bytes by multiplying `unit_mb` by 1 Megabyte (`MIB`).

`class SoftFilterConfig(_Strict):`
Defines the `SoftFilterConfig` model for rules that flag or filter out certain Wikipedia pages.

`    min_words: int | None = Field(default=50, ge=0)`
Requires pages to have at least 50 words by default, and must not be a negative number.

`    max_symbol_pct: int | None = Field(default=30, ge=0, le=100)`
Sets a rule where pages can't have more than 30% symbols by default, restricted between 0 and 100.

`    disambiguation: bool = True`
A true/false flag (defaulting to True) to handle disambiguation (multiple-meaning) pages.

`    list_page: bool = False`
A true/false flag (defaulting to False) to handle list pages.

`    def enabled(self) -> list[str]:`
Defines a method that returns a list of text strings representing which filters are turned on.

`        on = {`
Starts a dictionary called `on` to map filter names to whether they are active.

`            "min_words": self.min_words is not None,`
Checks if `min_words` has a value.

`            "max_symbol_pct": self.max_symbol_pct is not None,`
Checks if `max_symbol_pct` has a value.

`            "disambiguation": self.disambiguation,`
Checks if the disambiguation flag is True.

`            "list_page": self.list_page,`
Checks if the list_page flag is True.

`        }`
Closes the dictionary.

`        return sorted(name for name, flag in on.items() if flag)`
Loops through the dictionary and returns an alphabetically sorted list of the filter names that are currently active.

`class CleanConfig(_Strict):`
Defines the `CleanConfig` model for settings used to clean up the Wikipedia text.

`    drop_sections: list[str] = [`
Starts a list of Wikipedia section headers that should be removed from the text.

`        "References",`
Removes the "References" section.

`        "Notes",`
Removes the "Notes" section.

`        "External links",`
Removes the "External links" section.

`        "See also",`
Removes the "See also" section.

`        "Further reading",`
Removes the "Further reading" section.

`        "Bibliography",`
Removes the "Bibliography" section.

`        "Sources",`
Removes the "Sources" section.

`    ]`
Closes the list of sections to drop.

`    soft_filters: SoftFilterConfig = SoftFilterConfig()`
Includes the `SoftFilterConfig` settings inside this clean configuration, using its defaults.

`    disambiguation_templates: list[str] = ["disambiguation", "disambig", "dab", "hndis", "geodis"]`
Provides a list of common Wikipedia template names used to identify disambiguation pages.

`    language_codes_file: str = "configs/language_codes.txt"`
Specifies the default file path to find a list of valid language codes.

`class SelectConfig(_Strict):`
Defines the `SelectConfig` model to choose which cleaned pages are kept for the final dataset.

`    policy: str = Field(default="default", pattern=r"^[A-Za-z0-9_-]+$")`
Requires a policy name (defaulting to "default") containing only letters, numbers, underscores, or hyphens.

`    apply: list[str] = ["min_words", "disambiguation"]`
Specifies which soft filters (by default, "min_words" and "disambiguation") to apply when selecting pages.

`    dedup: Literal["exact"] = "exact"`
Sets the deduplication strategy to exactly "exact", meaning it will only remove exact duplicates.

`    val_per_10k: int = Field(default=50, ge=0, le=10_000)`
Sets how many pages out of every 10,000 should be held back for the validation set (default is 50).

`class PackConfig(_Strict):`
Defines the `PackConfig` model, detailing how the text data is packed together into files.

`    mode: Literal["concat", "bfd"] = "concat"`
Sets the packing mode to either "concat" (concatenation) or "bfd" (best-fit decreasing), defaulting to "concat".

`    seq_len: int = Field(ge=1)`
Requires a sequence length (the number of tokens in a batch) that must be at least 1.

`    last_row: Literal["drop", "pad"] | None = None  # concat default: drop; bfd always pads`
Allows defining what happens to the last row of data (drop it or pad it with empty space), defaulting to None.

`    shard_mb: int = Field(default=256, ge=1)`
Sets the target file size for data shards to 256 Megabytes, which must be at least 1.

`    shard_bytes: int | None = Field(default=None, ge=1)  # overrides shard_mb; for tiny fixtures`
An optional byte-level override for the shard size, mainly used for small tests.

`    bfd_window: int = Field(default=100_000, ge=1)`
Sets a window size of 100,000 for the "bfd" packing algorithm, meaning it looks at 100,000 items at a time.

`    @model_validator(mode="after")`
Tells Pydantic to run the following validation function after all fields are initially checked.

`    def _last_row(self) -> PackConfig:`
Defines a function to enforce specific rules about how the last row is handled.

`        if self.mode == "bfd" and self.last_row == "drop":`
Checks if the user set the mode to "bfd" but incorrectly told the system to "drop" the last row.

`            raise ValueError("last_row: drop applies to concat only; bfd always pads")`
Throws an error explaining that the "bfd" mode is not allowed to drop the last row.

`        return self`
Returns the validated object if no errors were found.

`    @property`
Creates a property so it can be accessed like an attribute.

`    def row_len(self) -> int:`
Defines a function to calculate the total row length.

`        return self.seq_len + 1`
Returns the sequence length plus 1 (often used in machine learning for the "next token" prediction).

`    @property`
Creates another property.

`    def effective_last_row(self) -> Literal["drop", "pad"]:`
Defines a function to determine the final, enforced rule for handling the last row.

`        if self.mode == "bfd":`
Checks if the mode is "bfd".

`            return "pad"`
If so, forces the last row behavior to be "pad".

`        return self.last_row or "drop"`
Otherwise, returns whatever the user set, or defaults to "drop".

`    @property`
Creates another property.

`    def target_shard_bytes(self) -> int:`
Defines a function to calculate the exact target size for data shards in bytes.

`        return self.shard_bytes if self.shard_bytes is not None else self.shard_mb * MIB`
Returns the explicit `shard_bytes` if provided, otherwise calculates it from `shard_mb`.

### Main Configuration Model

`class BuildConfig(_Strict):`
Defines the master `BuildConfig` model which pulls all the previous configuration pieces together.

`    name: str = Field(pattern=r"^[A-Za-z0-9._-]+$")`
Requires a build name using only letters, numbers, dots, underscores, or hyphens.

`    source: SourceConfig`
Requires a complete `SourceConfig` object.

`    workdir: str`
Requires a string defining the working directory path for temporary files.

`    output: str`
Requires a string defining the final output directory path.

`    workers: int = Field(default=1, ge=1)`
Specifies how many processor cores (workers) to use, defaulting to 1.

`    extract: ExtractConfig = ExtractConfig()`
Includes the `ExtractConfig` settings, using its defaults if not provided.

`    clean: CleanConfig = CleanConfig()`
Includes the `CleanConfig` settings, using its defaults if not provided.

`    select: SelectConfig = SelectConfig()`
Includes the `SelectConfig` settings, using its defaults if not provided.

`    tokenizer: str = Field(pattern=r"^(tiktoken|hf):.+$")`
Requires a tokenizer (text splitter) string that must start with either "tiktoken:" or "hf:".

`    pack: PackConfig`
Requires a complete `PackConfig` object.

`    @model_validator(mode="after")`
Tells Pydantic to run the following validation function after creating the build config.

`    def _cross_checks(self) -> BuildConfig:`
Defines a function to ensure the different configuration sections make sense together.

`        enabled = set(self.clean.soft_filters.enabled())`
Gets a mathematical set of all the soft filters that are actually turned on.

`        unknown = sorted(set(self.select.apply) - enabled)`
Finds any filters that the user asked to apply in `SelectConfig`, but which were not enabled in `CleanConfig`.

`        if unknown:`
If there are any unknown/un-enabled filters found...

`            raise ValueError(f"select.apply names filters that clean does not compute: {unknown}")`
Throws an error listing the problematic filters.

`        return self`
Returns the validated build configuration object.

`    def hashed_view(self) -> dict[str, Any]:`
Defines a method that creates a dictionary meant for calculating a unique fingerprint (hash) of the configuration.

`        """The output-affecting part of the config (PLAN A1).`
A comment explaining this method extracts only the settings that physically change the final output data.

`        Excludes workers, workdir, output and source.mirror, which never change output bytes,`
Explains that local paths and CPU counts are ignored because they don't change the data itself.

`        and replaces file paths with the SHA-256 of their content.`
Explains that local files referenced in the config are replaced by a hash of their contents to ensure reproducibility.

`        """`
Ends the comment block.

`        data = self.model_dump(mode="python")`
Converts the Pydantic model into a standard Python dictionary.

`        for key in ("workers", "workdir", "output"):`
Loops through the keys that shouldn't affect the data output.

`            data.pop(key)`
Removes those keys from the dictionary.

`        data["source"].pop("mirror")`
Removes the download mirror URL from the source dictionary.

`        data["clean"]["language_codes_file"] = sha256_file(self.clean.language_codes_file)`
Reads the language codes file and replaces the file path string with a cryptographic hash of the file's contents.

`        if self.tokenizer.startswith("hf:"):`
Checks if a Hugging Face ("hf:") local tokenizer file is being used.

`            data["tokenizer"] = "hf:sha256:" + sha256_file(self.tokenizer[3:])`
Replaces the tokenizer file path with a hash of that file's contents.

`        return data`
Returns the modified dictionary ready for hashing.

`    def config_sha256(self) -> str:`
Defines a method to generate the final configuration fingerprint.

`        return sha256_hex(canonical_json(self.hashed_view()))`
Converts the hashed view dictionary into a strict JSON string, hashes it, and returns the hexadecimal string.

### Mixture Models

`class MixSource(_Strict):`
Defines a model representing a single dataset source that will be mixed with others.

`    name: str = Field(pattern=r"^[A-Za-z0-9._-]+$")`
Requires a name for the source containing only letters, numbers, dots, underscores, or hyphens.

`    path: str`
Requires a string defining the file path to this dataset.

`    weight: int = Field(ge=1)`
Requires a mixing weight (how much of this data to use compared to others) that is at least 1.

`class MixConfig(_Strict):`
Defines a configuration model for mixing multiple datasets together.

`    seed: int = Field(ge=0, lt=1 << 64)`
Requires a random seed (for reproducibility) that must be positive and less than a 64-bit maximum.

`    split: Literal["train", "val"] = "train"`
Specifies whether this mix is for training data or validation data, defaulting to "train".

`    sources: list[MixSource] = Field(min_length=1)`
Requires a list of `MixSource` objects, and the list must contain at least one item.

`    @model_validator(mode="after")`
Tells Pydantic to run validation after all fields are set.

`    def _unique(self) -> MixConfig:`
Defines a function to ensure there are no duplicate sources in the mix.

`        names = [s.name for s in self.sources]`
Extracts a list of all the source names provided.

`        if len(set(names)) != len(names):`
Checks if converting the names to a set (which removes duplicates) makes the list shorter, meaning duplicates existed.

`            raise ValueError(f"duplicate source names: {names}")`
Throws an error if duplicates were found.

`        return self`
Returns the validated mix configuration object.

### File Loading Functions

`def _load_yaml(path: Path | str) -> Any:`
Defines a helper function to read a YAML file from a given path and return its contents.

`    try:`
Starts a block to attempt reading the file, catching errors if they happen.

`        with open(path, encoding="utf-8") as f:`
Opens the file safely in UTF-8 text mode, assigning it to the variable `f`.

`            return yaml.safe_load(f)`
Reads and parses the YAML file securely, returning the resulting Python dictionary.

`    except (OSError, yaml.YAMLError) as exc:`
Catches errors related to finding/reading the file (`OSError`) or parsing bad YAML formatting (`yaml.YAMLError`).

`        raise ConfigError(f"cannot read {path}: {exc}") from exc`
Wraps the system error into the custom `ConfigError` and throws it, preserving the original error trace.

`def load_build_config(path: Path | str) -> BuildConfig:`
Defines a function to read a YAML file specifically and convert it into a `BuildConfig` object.

`    try:`
Starts a block to attempt the conversion.

`        return BuildConfig.model_validate(_load_yaml(path))`
Loads the YAML file using the helper function, validates it against the `BuildConfig` model, and returns it.

`    except ValueError as exc:  # pydantic.ValidationError is a ValueError`
Catches validation errors (like missing fields or wrong data types) raised by Pydantic.

`        raise ConfigError(f"{path}: {exc}") from exc`
Wraps the validation error into the custom `ConfigError` and throws it.

`def load_mix_config(path: Path | str) -> MixConfig:`
Defines a function to read a YAML file specifically and convert it into a `MixConfig` object.

`    try:`
Starts a block to attempt the conversion.

`        return MixConfig.model_validate(_load_yaml(path))`
Loads the YAML file, validates it against the `MixConfig` model, and returns it.

`    except ValueError as exc:`
Catches validation errors raised by Pydantic.

`        raise ConfigError(f"{path}: {exc}") from exc`
Wraps the validation error into the custom `ConfigError` and throws it.

### Step 1.5: Tokenizer protocol and fingerprints

`src/wiki2shards/tokenize/base.py`

```python
"""Tokenizer protocol and fingerprints. Frozen at contracts-v1.

Spec strings: "tiktoken:<encoding name>" or "hf:<path to tokenizer.json>" (CWD-relative).
"""

```

These lines form a multi-line comment (called a docstring) that explains the purpose of the file. It states that this code defines the rules for text tokenizers and how to create unique identifiers (fingerprints) for them. It also notes that the code is stable ("Frozen") and shows the expected text formats for loading specific tokenizers.

```python
from __future__ import annotations

```

This enables a modern Python feature that allows type hints (labels telling the code what type of data to expect) to be read as plain text before everything in the file is fully loaded, preventing setup errors.

```python
import hashlib
import importlib
import json
import struct

```

These four lines bring in standard Python toolkits. `hashlib` is used to create secure, unique digital codes. `importlib` allows the program to load other Python files while it is running. `json` provides tools to read and write structured text data files. `struct` provides tools to pack basic data into raw binary formats.

```python
from collections.abc import Mapping, Sequence
from typing import Protocol, runtime_checkable

```

These lines import special tools used to describe how data should be shaped. `Mapping` represents dictionary-like structures (key-value pairs), and `Sequence` represents lists. `Protocol` is used to create a blueprint for how a piece of code should act, and `runtime_checkable` allows the program to verify if an object follows that blueprint while it is running.

```python
import numpy as np
from numpy.typing import NDArray

```

These lines import a popular math library called `numpy` (giving it the shorthand name `np`) and import a specific type label `NDArray`, which represents a numpy array (a highly efficient list used for numbers).

```python
from wiki2shards.io.hashing import canonical_json, sha256_hex

```

This line goes into another file in the same project and grabs two specific helper functions: one to format JSON data exactly the same way every time (`canonical_json`), and another to generate a specific type of unique security code (`sha256_hex`).

```python
EOS_TEXT = "<|endoftext|>"
PAD_TEXT = "<|pad|>"

```

These lines create two permanent labels (constants) that hold special text strings. `EOS_TEXT` represents the exact marker used to signal the end of a text block. `PAD_TEXT` represents the marker used to fill empty space so multiple pieces of text can be made exactly the same length.

```python
@runtime_checkable
class Tokenizer(Protocol):

```

This sets up a mandatory blueprint called `Tokenizer`. The `@runtime_checkable` tag means the code can actively verify if an object meets all the requirements of this blueprint. Any actual tokenizer tool used later in the code must contain everything listed inside this class.

```python
    @property
    def name(self) -> str: ...  # the spec string; goes to manifest tokenizer.spec

```

This dictates that any tokenizer must have a property called `name` that provides a text string. The comment notes this string is used in a record file (manifest). The `...` just means the actual tokenizer will fill in the details.

```python
    @property
    def fingerprint(self) -> str: ...  # sha256 hex

```

This dictates that the tokenizer must have a property called `fingerprint` that provides a unique text string (a 64-character code) to identify it.

```python
    @property
    def vocab_size(self) -> int: ...

```

This dictates that the tokenizer must have a property called `vocab_size` that provides a whole number (integer) representing how many distinct text pieces (tokens) it knows.

```python
    @property
    def eos_id(self) -> int: ...

```

This dictates that the tokenizer must have a property called `eos_id` that provides the specific whole number corresponding to the "end of text" marker.

```python
    @property
    def pad_id(self) -> int | None: ...

```

This dictates that the tokenizer must have a property called `pad_id` that provides either a whole number or nothing (`None`), representing the padding marker.

```python
    def encode_batch(self, texts: Sequence[str]) -> list[NDArray[np.int64]]:
        """Encode plain text. Special-token strings inside texts are ordinary text."""
        ...

```

This dictates that the tokenizer must have an `encode_batch` function. It must accept a list of text strings and return a list of numerical arrays. The comment clarifies that if any special markers appear naturally in the user's text, they should just be treated as regular text.

```python
    def decode(self, ids: Sequence[int] | NDArray[np.integer]) -> str: ...

```

This dictates that the tokenizer must have a `decode` function that accepts a list or array of whole numbers and converts them back into a single readable text string.

```python
def effective_pad_id(tok: Tokenizer) -> int:
    """GPT-2 has no pad token, so pad = eos; padding is identified by segments, not values."""
    return tok.pad_id if tok.pad_id is not None else tok.eos_id

```

This defines a new function that takes a tokenizer object. The comment explains that some models, like GPT-2, don't have a dedicated padding marker. The function checks if a padding ID exists; if it does, it returns it. If it doesn't, it returns the end-of-text ID instead.

```python
def fingerprint_hf_json(raw: bytes) -> str:
    """sha256 of the canonical form of tokenizer.json (floats are rejected: BPE files have none)."""
    return sha256_hex(canonical_json(json.loads(raw)))

```

This defines a function that takes raw binary data. It converts the binary data into a JSON object, formats it perfectly cleanly, and then generates a unique identifier code from it. This is used to securely identify Hugging Face ("hf") format tokenizers.

```python
def fingerprint_tiktoken(
    name: str,
    pat_str: str,
    special_tokens: Mapping[str, int],
    mergeable_ranks: Mapping[bytes, int],
) -> str:

```

This starts defining a function to create a unique identifier for OpenAI ("tiktoken") tokenizers. It requires four inputs: a name, a text pattern, a dictionary of special markers, and a dictionary of standard text fragments paired with their numerical ranks. It promises to output a text string.

```python
    """sha256 over a canonical header, then (<u4 rank, <u4 len, bytes) for each rank in order."""

```

This is a comment explaining exactly how the unique identifier will be mathematically constructed piece by piece.

```python
    h = hashlib.sha256()

```

This creates a fresh, empty digital blender (a sha256 hashing object) to start mixing the data together securely.

```python
    header = {"name": name, "pat_str": pat_str, "special_tokens": dict(special_tokens)}

```

This groups the name, pattern string, and special tokens together into a single dictionary called a `header`.

```python
    h.update(canonical_json(header) + b"\n")

```

This converts the `header` dictionary into a perfectly formatted text byte string, adds a "new line" character to the end, and drops it into the hashing object.

```python
    for token, rank in sorted(mergeable_ranks.items(), key=lambda kv: kv[1]):

```

This line starts a loop that goes through every standard token fragment. Before looping, it sorts them by their numerical rank from lowest to highest.

```python
        h.update(struct.pack("<II", rank, len(token)))

```

For every token in the loop, this packs its numerical rank and the length of its text into a tiny chunk of raw binary data and drops it into the hashing object.

```python
        h.update(token)

```

This then drops the actual raw text bytes of the token itself into the hashing object.

```python
    return h.hexdigest()

```

Once the loop is finished, this asks the hashing object to blend all the dropped-in pieces and return the final unique identifier as a readable text string.

```python
_CACHE: dict[str, Tokenizer] = {}

```

This creates an empty dictionary named `_CACHE`. The underscore means it is meant for internal use only. This acts as a memory bank to save tokenizers once they are loaded so the program doesn't have to rebuild them if asked for them again.

```python
def load_tokenizer(spec: str) -> Tokenizer:
    """Construct (once per process) the tokenizer named by a spec string.

    Adapters are imported by name, so this contract module does not depend on them (they are
    written later, in tokenize/hf_tok.py and tokenize/tiktoken_tok.py).
    """

```

This begins a function that takes a specification text string and promises to build and return a fully working Tokenizer. The comment explains that the actual building tools are loaded on the fly from other files, keeping this core file independent.

```python
    if spec in _CACHE:
        return _CACHE[spec]

```

This checks if the requested tokenizer has already been built and saved in the memory bank (`_CACHE`). If it has, it immediately returns the saved version to save time.

```python
    kind, _, arg = spec.partition(":")

```

This splits the requested specification text string into three pieces at the colon. The word before the colon becomes `kind` (e.g., "tiktoken"), the colon itself is ignored (the `_`), and the rest becomes `arg` (e.g., "gpt2").

```python
    tok: Tokenizer

```

This tells Python to prepare a variable named `tok` that will eventually hold an object matching the Tokenizer blueprint.

```python
    if kind == "tiktoken":

```

This checks if the first half of the requested string was "tiktoken".

```python
        tok = importlib.import_module(
            "wiki2shards.tokenize.tiktoken_tok"
        ).TiktokenAdapter.from_name(arg)

```

If it was "tiktoken", the program opens a file named `tiktoken_tok` inside the project, grabs a specific tool inside called `TiktokenAdapter`, and uses it to build the tokenizer based on the second half of the requested string (`arg`). It assigns the result to `tok`.

```python
    elif kind == "hf":

```

If the first half of the requested string wasn't "tiktoken", this checks if it was "hf" (Hugging Face).

```python
        tok = importlib.import_module("wiki2shards.tokenize.hf_tok").HFAdapter.from_file(
            arg, spec=spec
        )

```

If it was "hf", the program opens a file named `hf_tok`, grabs a tool called `HFAdapter`, and uses it to build the tokenizer by reading a file located at the path provided in `arg`. It assigns the result to `tok`.

```python
    else:
        raise ValueError(f"unknown tokenizer spec {spec!r}")

```

If the string didn't start with "tiktoken" or "hf", the program stops working and throws an error message explaining that it doesn't recognize the request format.

```python
    _CACHE[spec] = tok

```

After successfully building the tokenizer, this saves a copy of it into the memory bank (`_CACHE`), using the original request string as the label.

```python
    return tok

```

Finally, this hands the newly built tokenizer back to whatever part of the program originally asked for it.